In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, '../src')

from ne_aligned_utils.benchmarking import (
    build_four_block_matrices,
    build_metric_heatmap_data,
    compute_shared_dataset_scale_bounds,
    compute_eot_gmm_rand,
    compute_gmm_rand_summary,
    compute_eot_reference_from_plans,
    compute_ot_plans_for_eps,
    load_benchmark_cache,
    load_eot_reference_cache,
    make_eps_cache_path,
    make_eps_eot_cache_path,
    plot_gmm_pct_embedding_grid_by_cluster,
    plot_gmm_rand_summary,
    plot_k50_concordance_summary,
    plot_metric_heatmap_grid,
    plot_ot_plan_heatmaps,
    plot_pct_embedding_grid,
    prepare_benchmark_data,
    run_top_pct_neighbors_benchmark,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True

def load_cached_benchmark_results(cache_dir, eps_times):
    cache_path = make_eps_cache_path(cache_dir, eps_times)
    if not cache_path.exists():
        raise FileNotFoundError(
            f"No saved benchmark cache found at {cache_path}. Run the benchmark cell first."
        )
    return load_benchmark_cache(cache_path)

def load_cached_eot_reference(cache_dir, eps_times):
    cache_path = make_eps_eot_cache_path(cache_dir, eps_times)
    if not cache_path.exists():
        raise FileNotFoundError(
            f"No saved EOT reference cache found at {cache_path}. Run the EOT cell first."
        )
    return load_eot_reference_cache(cache_path)


In [ ]:
seed = 0

pct_values = [0.1, 0.5, 1, 5, 10, 12, 15, 17, 20, 22, 25, 30, 35, 40]
eps_times_values = [1, 10, 100, 1000, 10000, 100000]

tsne_seeds = [0, 1, 2, 3]
viz_seed = 0
n_epochs = 200
k_conc = 50
k_cross = 50
eot_t = 0
eot_embd_dim = 2

tol = 1e-12
max_iter = int(1e4)

n0_ring = 1000
n1_ring = 5000

gmm_n0 = 600
gmm_n1 = 600
gmm_p = 1000
gmm_q = 6
gmm_theta = 3.0

cache_dir = Path('../data/benchmarking')
cache_dir.mkdir(parents=True, exist_ok=True)

if viz_seed not in tsne_seeds:
    raise ValueError(f'viz_seed={viz_seed} must be included in tsne_seeds={tsne_seeds}')


## Create Data


In [ ]:
data_bundle = prepare_benchmark_data(
    seed=seed,
    n0_ring=n0_ring,
    n1_ring=n1_ring,
    gmm_n0=gmm_n0,
    gmm_n1=gmm_n1,
    gmm_p=gmm_p,
    gmm_q=gmm_q,
    gmm_theta=gmm_theta,
)

fig = plt.figure(figsize=(12, 5.5))
ax_ring = fig.add_subplot(121, projection='3d')
ax_ring.scatter(*data_bundle['ring']['obs0'].T, c='tab:red', s=6, alpha=0.8, label='dataset 0')
ax_ring.scatter(*data_bundle['ring']['obs1'].T, c='tab:blue', s=3, alpha=0.6, label='dataset 1')
ax_ring.set_title('Ring data (current make_fig1_data)')
ax_ring.view_init(elev=30, azim=-60)
ax_ring.legend(markerscale=2, loc='upper left')

ax_gmm = fig.add_subplot(122)
ax_gmm.scatter(
    data_bundle['gmm']['obs0'][:, 0],
    data_bundle['gmm']['obs0'][:, 1],
    s=6,
    c='tab:red',
    alpha=0.6,
    label='dataset 0',
)
ax_gmm.scatter(
    data_bundle['gmm']['obs1'][:, 0],
    data_bundle['gmm']['obs1'][:, 1],
    s=6,
    c='tab:blue',
    alpha=0.6,
    label='dataset 1',
)
ax_gmm.set_title('GMM data')
ax_gmm.set_xlabel('dim 1')
ax_gmm.set_ylabel('dim 2')
ax_gmm.set_aspect('equal')
ax_gmm.legend(loc='best')

plt.tight_layout()
plt.show()

print(
    f"ring observed data: x0={data_bundle['ring']['obs0'].shape}, x1={data_bundle['ring']['obs1'].shape}; "
    f"ring clean data: x0={data_bundle['ring']['clean0'].shape}, x1={data_bundle['ring']['clean1'].shape}"
)
print(
    f"gmm observed data: x0={data_bundle['gmm']['obs0'].shape}, x1={data_bundle['gmm']['obs1'].shape}; "
    f"gmm clean data: x0={data_bundle['gmm']['clean0'].shape}, x1={data_bundle['gmm']['clean1'].shape}"
)
print(
    f"GMM sort indices: idx0={data_bundle['gmm']['sort_idx0'].shape}, idx1={data_bundle['gmm']['sort_idx1'].shape}"
)


## eps = 1


In [ ]:
eps_times_1 = 1
plans_1 = compute_ot_plans_for_eps(
    data_bundle,
    eps_times=eps_times_1,
    tol=tol,
    max_iter=max_iter,
)

for dataset_name in ('ring', 'gmm'):
    plan = plans_1[dataset_name]['P']
    print(f"{dataset_name}: shape={plan.shape}, sum={plan.sum():.6f}")

fig, axes = plot_ot_plan_heatmaps(plans_1, eps_times=eps_times_1)
plt.show()


In [ ]:
eot_cache_path_1 = make_eps_eot_cache_path(cache_dir, eps_times_1)
eot_reference_1 = compute_eot_reference_from_plans(
    data_bundle,
    plans_1,
    t=eot_t,
    embd_dim=eot_embd_dim,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=eot_cache_path_1,
)

print(f"Saved EOT reference cache to {eot_cache_path_1}")

for dataset_name in ('ring', 'gmm'):
    metrics = eot_reference_1[dataset_name]['metrics']
    print(
        f"{dataset_name} EOT t={eot_reference_1['t']}: "
        f"conc={metrics['concordance']:.4f}, "
        f"xconc={metrics['cross_concordance']:.4f}, "
        f"foscttm={metrics['foscttm']:.4f}"
    )


In [ ]:
block_matrices_1 = build_four_block_matrices(plans_1)
cache_path_1 = make_eps_cache_path(cache_dir, eps_times_1)
benchmark_results_1 = run_top_pct_neighbors_benchmark(
    data_bundle,
    block_matrices_1,
    pct_values=pct_values,
    tsne_seeds=tsne_seeds,
    viz_seed=viz_seed,
    n_epochs=n_epochs,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=cache_path_1,
)

print(f"Saved benchmark cache to {cache_path_1}")
print(
    f"ring block matrix: shape={block_matrices_1['ring']['matrix'].shape}, "
    f"sum={block_matrices_1['ring']['matrix'].sum():.6f}"
)
print(
    f"gmm block matrix: shape={block_matrices_1['gmm']['matrix'].shape}, "
    f"sum={block_matrices_1['gmm']['matrix'].sum():.6f}"
)


In [ ]:
if 'benchmark_results_1' not in globals():
    benchmark_results_1 = load_cached_benchmark_results(cache_dir, eps_times_1)

fig, axes = plot_pct_embedding_grid(
    benchmark_results_1,
    data_bundle,
    title=(
        f"Top % Neighbors embeddings (eps_times={benchmark_results_1['eps_times']:g}, "
        f"viz seed={benchmark_results_1['viz_seed']})"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_1' not in globals():
    benchmark_results_1 = load_cached_benchmark_results(cache_dir, eps_times_1)

fig, axes = plot_gmm_pct_embedding_grid_by_cluster(
    benchmark_results_1,
    data_bundle,
    title=(
        f"Top % Neighbors GMM embeddings (eps_times={benchmark_results_1['eps_times']:g}, "
        f"viz seed={benchmark_results_1['viz_seed']}, colored by cluster)"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_1' not in globals():
    benchmark_results_1 = load_cached_benchmark_results(cache_dir, eps_times_1)
if 'eot_reference_1' not in globals():
    eot_reference_1 = load_cached_eot_reference(cache_dir, eps_times_1)

fig, axes = plot_k50_concordance_summary(
    benchmark_results_1,
    eot_reference_1,
    title=(
        f"k=50 concordance summary (eps_times={benchmark_results_1['eps_times']:g}, "
        f"EOT t={eot_reference_1['t']})"
    ),
)
plt.show()


## eps = 10


In [ ]:
eps_times_10 = 10
plans_10 = compute_ot_plans_for_eps(
    data_bundle,
    eps_times=eps_times_10,
    tol=tol,
    max_iter=max_iter,
)

for dataset_name in ('ring', 'gmm'):
    plan = plans_10[dataset_name]['P']
    print(f"{dataset_name}: shape={plan.shape}, sum={plan.sum():.6f}")

fig, axes = plot_ot_plan_heatmaps(plans_10, eps_times=eps_times_10)
plt.show()


In [ ]:
eot_cache_path_10 = make_eps_eot_cache_path(cache_dir, eps_times_10)
eot_reference_10 = compute_eot_reference_from_plans(
    data_bundle,
    plans_10,
    t=eot_t,
    embd_dim=eot_embd_dim,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=eot_cache_path_10,
)

print(f"Saved EOT reference cache to {eot_cache_path_10}")

for dataset_name in ('ring', 'gmm'):
    metrics = eot_reference_10[dataset_name]['metrics']
    print(
        f"{dataset_name} EOT t={eot_reference_10['t']}: "
        f"conc={metrics['concordance']:.4f}, "
        f"xconc={metrics['cross_concordance']:.4f}, "
        f"foscttm={metrics['foscttm']:.4f}"
    )


In [ ]:
block_matrices_10 = build_four_block_matrices(plans_10)
cache_path_10 = make_eps_cache_path(cache_dir, eps_times_10)
benchmark_results_10 = run_top_pct_neighbors_benchmark(
    data_bundle,
    block_matrices_10,
    pct_values=pct_values,
    tsne_seeds=tsne_seeds,
    viz_seed=viz_seed,
    n_epochs=n_epochs,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=cache_path_10,
)

print(f"Saved benchmark cache to {cache_path_10}")
print(
    f"ring block matrix: shape={block_matrices_10['ring']['matrix'].shape}, "
    f"sum={block_matrices_10['ring']['matrix'].sum():.6f}"
)
print(
    f"gmm block matrix: shape={block_matrices_10['gmm']['matrix'].shape}, "
    f"sum={block_matrices_10['gmm']['matrix'].sum():.6f}"
)


In [ ]:
if 'benchmark_results_10' not in globals():
    benchmark_results_10 = load_cached_benchmark_results(cache_dir, eps_times_10)

fig, axes = plot_pct_embedding_grid(
    benchmark_results_10,
    data_bundle,
    title=(
        f"Top % Neighbors embeddings (eps_times={benchmark_results_10['eps_times']:g}, "
        f"viz seed={benchmark_results_10['viz_seed']})"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_10' not in globals():
    benchmark_results_10 = load_cached_benchmark_results(cache_dir, eps_times_10)

fig, axes = plot_gmm_pct_embedding_grid_by_cluster(
    benchmark_results_10,
    data_bundle,
    title=(
        f"Top % Neighbors GMM embeddings (eps_times={benchmark_results_10['eps_times']:g}, "
        f"viz seed={benchmark_results_10['viz_seed']}, colored by cluster)"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_10' not in globals():
    benchmark_results_10 = load_cached_benchmark_results(cache_dir, eps_times_10)
if 'eot_reference_10' not in globals():
    eot_reference_10 = load_cached_eot_reference(cache_dir, eps_times_10)

fig, axes = plot_k50_concordance_summary(
    benchmark_results_10,
    eot_reference_10,
    title=(
        f"k=50 concordance summary (eps_times={benchmark_results_10['eps_times']:g}, "
        f"EOT t={eot_reference_10['t']})"
    ),
)
plt.show()


## eps = 100


In [ ]:
eps_times_100 = 100
plans_100 = compute_ot_plans_for_eps(
    data_bundle,
    eps_times=eps_times_100,
    tol=tol,
    max_iter=max_iter,
)

for dataset_name in ('ring', 'gmm'):
    plan = plans_100[dataset_name]['P']
    print(f"{dataset_name}: shape={plan.shape}, sum={plan.sum():.6f}")

fig, axes = plot_ot_plan_heatmaps(plans_100, eps_times=eps_times_100)
plt.show()


In [ ]:
eot_cache_path_100 = make_eps_eot_cache_path(cache_dir, eps_times_100)
eot_reference_100 = compute_eot_reference_from_plans(
    data_bundle,
    plans_100,
    t=eot_t,
    embd_dim=eot_embd_dim,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=eot_cache_path_100,
)

print(f"Saved EOT reference cache to {eot_cache_path_100}")

for dataset_name in ('ring', 'gmm'):
    metrics = eot_reference_100[dataset_name]['metrics']
    print(
        f"{dataset_name} EOT t={eot_reference_100['t']}: "
        f"conc={metrics['concordance']:.4f}, "
        f"xconc={metrics['cross_concordance']:.4f}, "
        f"foscttm={metrics['foscttm']:.4f}"
    )


In [ ]:
block_matrices_100 = build_four_block_matrices(plans_100)
cache_path_100 = make_eps_cache_path(cache_dir, eps_times_100)
benchmark_results_100 = run_top_pct_neighbors_benchmark(
    data_bundle,
    block_matrices_100,
    pct_values=pct_values,
    tsne_seeds=tsne_seeds,
    viz_seed=viz_seed,
    n_epochs=n_epochs,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=cache_path_100,
)

print(f"Saved benchmark cache to {cache_path_100}")
print(
    f"ring block matrix: shape={block_matrices_100['ring']['matrix'].shape}, "
    f"sum={block_matrices_100['ring']['matrix'].sum():.6f}"
)
print(
    f"gmm block matrix: shape={block_matrices_100['gmm']['matrix'].shape}, "
    f"sum={block_matrices_100['gmm']['matrix'].sum():.6f}"
)


In [ ]:
if 'benchmark_results_100' not in globals():
    benchmark_results_100 = load_cached_benchmark_results(cache_dir, eps_times_100)

fig, axes = plot_pct_embedding_grid(
    benchmark_results_100,
    data_bundle,
    title=(
        f"Top % Neighbors embeddings (eps_times={benchmark_results_100['eps_times']:g}, "
        f"viz seed={benchmark_results_100['viz_seed']})"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_100' not in globals():
    benchmark_results_100 = load_cached_benchmark_results(cache_dir, eps_times_100)

fig, axes = plot_gmm_pct_embedding_grid_by_cluster(
    benchmark_results_100,
    data_bundle,
    title=(
        f"Top % Neighbors GMM embeddings (eps_times={benchmark_results_100['eps_times']:g}, "
        f"viz seed={benchmark_results_100['viz_seed']}, colored by cluster)"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_100' not in globals():
    benchmark_results_100 = load_cached_benchmark_results(cache_dir, eps_times_100)
if 'eot_reference_100' not in globals():
    eot_reference_100 = load_cached_eot_reference(cache_dir, eps_times_100)

fig, axes = plot_k50_concordance_summary(
    benchmark_results_100,
    eot_reference_100,
    title=(
        f"k=50 concordance summary (eps_times={benchmark_results_100['eps_times']:g}, "
        f"EOT t={eot_reference_100['t']})"
    ),
)
plt.show()


## eps = 300


In [ ]:
eps_times_1000 = 300
plans_1000 = compute_ot_plans_for_eps(
    data_bundle,
    eps_times=eps_times_1000,
    tol=tol,
    max_iter=max_iter,
)

for dataset_name in ('ring', 'gmm'):
    plan = plans_1000[dataset_name]['P']
    print(f"{dataset_name}: shape={plan.shape}, sum={plan.sum():.6f}")

fig, axes = plot_ot_plan_heatmaps(plans_1000, eps_times=eps_times_1000)
plt.show()


In [ ]:
eot_cache_path_1000 = make_eps_eot_cache_path(cache_dir, eps_times_1000)
eot_reference_1000 = compute_eot_reference_from_plans(
    data_bundle,
    plans_1000,
    t=eot_t,
    embd_dim=eot_embd_dim,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=eot_cache_path_1000,
)

print(f"Saved EOT reference cache to {eot_cache_path_1000}")

for dataset_name in ('ring', 'gmm'):
    metrics = eot_reference_1000[dataset_name]['metrics']
    print(
        f"{dataset_name} EOT t={eot_reference_1000['t']}: "
        f"conc={metrics['concordance']:.4f}, "
        f"xconc={metrics['cross_concordance']:.4f}, "
        f"foscttm={metrics['foscttm']:.4f}"
    )


In [ ]:
block_matrices_1000 = build_four_block_matrices(plans_1000)
cache_path_1000 = make_eps_cache_path(cache_dir, eps_times_1000)
benchmark_results_1000 = run_top_pct_neighbors_benchmark(
    data_bundle,
    block_matrices_1000,
    pct_values=pct_values,
    tsne_seeds=tsne_seeds,
    viz_seed=viz_seed,
    n_epochs=n_epochs,
    k_conc=k_conc,
    k_cross=k_cross,
    cache_path=cache_path_1000,
)

print(f"Saved benchmark cache to {cache_path_1000}")
print(
    f"ring block matrix: shape={block_matrices_1000['ring']['matrix'].shape}, "
    f"sum={block_matrices_1000['ring']['matrix'].sum():.6f}"
)
print(
    f"gmm block matrix: shape={block_matrices_1000['gmm']['matrix'].shape}, "
    f"sum={block_matrices_1000['gmm']['matrix'].sum():.6f}"
)


In [ ]:
if 'benchmark_results_1000' not in globals():
    benchmark_results_1000 = load_cached_benchmark_results(cache_dir, eps_times_1000)

fig, axes = plot_pct_embedding_grid(
    benchmark_results_1000,
    data_bundle,
    title=(
        f"Top % Neighbors embeddings (eps_times={benchmark_results_1000['eps_times']:g}, "
        f"viz seed={benchmark_results_1000['viz_seed']})"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_1000' not in globals():
    benchmark_results_1000 = load_cached_benchmark_results(cache_dir, eps_times_1000)

fig, axes = plot_gmm_pct_embedding_grid_by_cluster(
    benchmark_results_1000,
    data_bundle,
    title=(
        f"Top % Neighbors GMM embeddings (eps_times={benchmark_results_1000['eps_times']:g}, "
        f"viz seed={benchmark_results_1000['viz_seed']}, colored by cluster)"
    ),
)
plt.show()


In [ ]:
if 'benchmark_results_1000' not in globals():
    benchmark_results_1000 = load_cached_benchmark_results(cache_dir, eps_times_1000)
if 'eot_reference_1000' not in globals():
    eot_reference_1000 = load_cached_eot_reference(cache_dir, eps_times_1000)

fig, axes = plot_k50_concordance_summary(
    benchmark_results_1000,
    eot_reference_1000,
    title=(
        f"k=50 concordance summary (eps_times={benchmark_results_1000['eps_times']:g}, "
        f"EOT t={eot_reference_1000['t']})"
    ),
)
plt.show()


## Across-Epsilon Metric Heatmaps


In [ ]:
if 'cache_dir' in globals():
    cache_dir = Path(cache_dir)
else:
    cache_dir_candidates = [Path('../data/benchmarking'), Path('data/benchmarking')]
    cache_dir = next(
        (
            path
            for path in cache_dir_candidates
            if list(path.glob('top_pct_neighbors_seed_sweep_eps_times_*.npz'))
        ),
        cache_dir_candidates[0],
    )
cache_dir.mkdir(parents=True, exist_ok=True)

benchmark_cache_paths = sorted(cache_dir.glob('top_pct_neighbors_seed_sweep_eps_times_*.npz'))
eot_cache_paths = sorted(cache_dir.glob('eot_reference_eps_times_*.npz'))

if not benchmark_cache_paths:
    raise FileNotFoundError(f'No benchmark cache files found in {cache_dir}')
if not eot_cache_paths:
    raise FileNotFoundError(f'No EOT reference cache files found in {cache_dir}')

eot_references_by_eps = {}
for eot_cache_path in eot_cache_paths:
    with np.load(eot_cache_path) as eot_npz:
        metric_names_in_cache = [str(name) for name in eot_npz['metric_names']]
        metric_idx = {name: i for i, name in enumerate(metric_names_in_cache)}
        eps = float(eot_npz['eps_times'])
        eot_references_by_eps[eps] = {
            't': int(eot_npz['t']),
            'ring': {
                'metrics': {
                    name: float(eot_npz['ring_metrics'][metric_idx[name]])
                    for name in metric_names_in_cache
                }
            },
            'gmm': {
                'metrics': {
                    name: float(eot_npz['gmm_metrics'][metric_idx[name]])
                    for name in metric_names_in_cache
                }
            },
        }

benchmark_results_by_eps = {}
for benchmark_cache_path in benchmark_cache_paths:
    with np.load(benchmark_cache_path) as benchmark_npz:
        metric_names_in_cache = [str(name) for name in benchmark_npz['metric_names']]
        metric_idx = {name: i for i, name in enumerate(metric_names_in_cache)}
        eps = float(benchmark_npz['eps_times'])
        results = {}
        for row_idx, pct in enumerate(benchmark_npz['pct_values']):
            results[float(pct)] = {
                'ring_mean': {
                    name: float(benchmark_npz['ring_metric_mean'][row_idx, metric_idx[name]])
                    for name in metric_names_in_cache
                },
                'gmm_mean': {
                    name: float(benchmark_npz['gmm_metric_mean'][row_idx, metric_idx[name]])
                    for name in metric_names_in_cache
                },
            }
        benchmark_results_by_eps[eps] = {
            'eps_times': eps,
            'pct_values': [float(pct) for pct in benchmark_npz['pct_values']],
            'results': results,
        }

available_eps = sorted(benchmark_results_by_eps)
missing_eot_eps = [eps for eps in available_eps if eps not in eot_references_by_eps]
if missing_eot_eps:
    raise KeyError(f'Missing EOT reference cache(s) for eps_times={missing_eot_eps}')

print(f'Loaded benchmark caches for eps_times={available_eps}')

figure_dir = cache_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
heatmap_cache_path = cache_dir / 'eps_pct_metric_heatmaps.npz'

metric_names = ('concordance', 'cross_concordance')
heatmap_data_by_metric = {}

for metric_name in metric_names:
    heatmap_data = build_metric_heatmap_data(
        benchmarks_by_eps=benchmark_results_by_eps,
        eot_references_by_eps=eot_references_by_eps,
        metric_name=metric_name,
    )
    heatmap_data_by_metric[metric_name] = heatmap_data

shared_scale_bounds = compute_shared_dataset_scale_bounds(heatmap_data_by_metric)

for metric_name in metric_names:
    heatmap_data = heatmap_data_by_metric[metric_name]

    figure_path = figure_dir / f'{metric_name}_eps_pct_heatmaps.png'
    fig, axes = plot_metric_heatmap_grid(
        heatmap_data,
        title=(
            f"{heatmap_data['metric_label']} Heatmaps"
        ),
        save_path=figure_path,
        scale_bounds_by_dataset=shared_scale_bounds,
        annotate_cells=True,
    )
    plt.show()
    print(f"Saved {metric_name} heatmap figure to {figure_path}")

np.savez_compressed(
    heatmap_cache_path,
    eps_times=heatmap_data_by_metric['concordance']['eps_times'],
    pct_values=heatmap_data_by_metric['concordance']['pct_values'],
    dataset_names=np.asarray(heatmap_data_by_metric['concordance']['dataset_names']),
    metric_names=np.asarray(metric_names),
    eot_t=np.asarray(heatmap_data_by_metric['concordance']['eot_t'], dtype=int),
    x_labels=np.asarray(heatmap_data_by_metric['concordance']['x_labels']),
    y_labels=np.asarray(heatmap_data_by_metric['concordance']['y_labels']),
    concordance_ring_full=heatmap_data_by_metric['concordance']['datasets']['ring']['full_matrix'],
    concordance_ring_top_pct=heatmap_data_by_metric['concordance']['datasets']['ring']['top_pct_matrix'],
    concordance_ring_eot=heatmap_data_by_metric['concordance']['datasets']['ring']['eot_column'],
    concordance_gmm_full=heatmap_data_by_metric['concordance']['datasets']['gmm']['full_matrix'],
    concordance_gmm_top_pct=heatmap_data_by_metric['concordance']['datasets']['gmm']['top_pct_matrix'],
    concordance_gmm_eot=heatmap_data_by_metric['concordance']['datasets']['gmm']['eot_column'],
    cross_concordance_ring_full=heatmap_data_by_metric['cross_concordance']['datasets']['ring']['full_matrix'],
    cross_concordance_ring_top_pct=heatmap_data_by_metric['cross_concordance']['datasets']['ring']['top_pct_matrix'],
    cross_concordance_ring_eot=heatmap_data_by_metric['cross_concordance']['datasets']['ring']['eot_column'],
    cross_concordance_gmm_full=heatmap_data_by_metric['cross_concordance']['datasets']['gmm']['full_matrix'],
    cross_concordance_gmm_top_pct=heatmap_data_by_metric['cross_concordance']['datasets']['gmm']['top_pct_matrix'],
    cross_concordance_gmm_eot=heatmap_data_by_metric['cross_concordance']['datasets']['gmm']['eot_column'],
)

print(f"Saved heatmap matrices to {heatmap_cache_path}")


## GMM Rand Index


In [ ]:
if 'cache_dir' in globals():
    rand_cache_dir = Path(cache_dir)
else:
    rand_cache_candidates = [Path('../data/benchmarking'), Path('data/benchmarking')]
    rand_cache_dir = next(
        (
            path
            for path in rand_cache_candidates
            if list(path.glob('top_pct_neighbors_seed_sweep_eps_times_*.npz'))
        ),
        rand_cache_candidates[0],
    )
rand_cache_dir.mkdir(parents=True, exist_ok=True)

if 'data_bundle' not in globals():
    data_bundle = prepare_benchmark_data(
        seed=globals().get('seed', 0),
        n0_ring=globals().get('n0_ring', 1000),
        n1_ring=globals().get('n1_ring', 5000),
        gmm_n0=globals().get('gmm_n0', 600),
        gmm_n1=globals().get('gmm_n1', 600),
        gmm_p=globals().get('gmm_p', 1000),
        gmm_q=globals().get('gmm_q', 6),
        gmm_theta=globals().get('gmm_theta', 3.0),
    )

gmm_gt_clusters = np.concatenate([data_bundle['gmm']['clusters0'], data_bundle['gmm']['clusters1']])
rand_cluster_seed = 0

benchmark_cache_paths = sorted(rand_cache_dir.glob('top_pct_neighbors_seed_sweep_eps_times_*.npz'))
eot_cache_paths = sorted(rand_cache_dir.glob('eot_reference_eps_times_*.npz'))

if not benchmark_cache_paths:
    raise FileNotFoundError(f'No benchmark cache files found in {rand_cache_dir}')
if not eot_cache_paths:
    raise FileNotFoundError(f'No EOT reference cache files found in {rand_cache_dir}')

eot_references_by_eps = {}
for eot_cache_path in eot_cache_paths:
    with np.load(eot_cache_path) as eot_npz:
        eps = float(eot_npz['eps_times'])
        eot_references_by_eps[eps] = {
            't': int(eot_npz['t']),
            'gmm': {'embedding': np.asarray(eot_npz['gmm_embedding'])},
        }

benchmark_results_by_eps = {}
for benchmark_cache_path in benchmark_cache_paths:
    with np.load(benchmark_cache_path) as benchmark_npz:
        eps = float(benchmark_npz['eps_times'])
        pct_values_loaded = [float(pct) for pct in benchmark_npz['pct_values']]
        tsne_seeds_loaded = [int(seed) for seed in benchmark_npz['tsne_seeds']]
        results = {}
        for row_idx, pct in enumerate(pct_values_loaded):
            results[float(pct)] = {
                'gmm_embeddings_by_seed': {
                    int(seed): np.asarray(benchmark_npz['gmm_embeddings'][row_idx, seed_idx])
                    for seed_idx, seed in enumerate(tsne_seeds_loaded)
                }
            }
        benchmark_results_by_eps[eps] = {
            'eps_times': eps,
            'pct_values': pct_values_loaded,
            'tsne_seeds': tsne_seeds_loaded,
            'results': results,
        }

available_eps = sorted(benchmark_results_by_eps)
missing_eot_eps = [eps for eps in available_eps if eps not in eot_references_by_eps]
if missing_eot_eps:
    raise KeyError(f'Missing EOT reference cache(s) for eps_times={missing_eot_eps}')

figure_dir = rand_cache_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
rand_cache_path = rand_cache_dir / 'gmm_rand_eps_pct_heatmap.npz'

gmm_rand_summary_by_eps = {}
gmm_eot_rand_by_eps = {}

for eps in available_eps:
    benchmark_results = benchmark_results_by_eps[eps]
    eot_reference = eot_references_by_eps[eps]
    fig, ax, rand_summary, eot_rand = plot_gmm_rand_summary(
        benchmark_results,
        eot_reference,
        gt_clusters=gmm_gt_clusters,
        cluster_seed=rand_cluster_seed,
        title=(
            f"GMM Rand index summary (eps_times={eps:g}, "
            f"EOT t={eot_reference['t']})"
        ),
    )
    plt.show()

    eps_token = f'{eps:g}'.replace('-', 'm').replace('.', 'p')
    summary_path = figure_dir / f'gmm_rand_summary_eps_times_{eps_token}.png'
    fig.savefig(summary_path, bbox_inches='tight')
    print(f"Saved GMM Rand summary figure to {summary_path}")

    gmm_rand_summary_by_eps[eps] = rand_summary
    gmm_eot_rand_by_eps[eps] = eot_rand

rand_eps_values = np.array(available_eps, dtype=float)
rand_pct_values = np.asarray(gmm_rand_summary_by_eps[float(rand_eps_values[0])]['pct_values'], dtype=float)
rand_heatmap = np.stack([gmm_rand_summary_by_eps[eps]['mean'] for eps in rand_eps_values], axis=0)
rand_eot = np.asarray([gmm_eot_rand_by_eps[eps] for eps in rand_eps_values], dtype=float)
rand_eot_t = int(next(iter(eot_references_by_eps.values()))['t'])

rand_heatmap_data = {
    'metric_name': 'rand',
    'metric_label': 'Rand Index',
    'eps_times': rand_eps_values,
    'pct_values': rand_pct_values,
    'eot_t': rand_eot_t,
    'dataset_names': ('gmm',),
    'datasets': {
        'gmm': {
            'top_pct_matrix': rand_heatmap,
            'eot_column': rand_eot,
        }
    },
    'y_labels': [f'{eps:g}' for eps in rand_eps_values],
    'top_pct_column_count': int(rand_pct_values.size),
}

rand_heatmap_path = figure_dir / 'gmm_rand_eps_pct_heatmap.png'
fig, axes = plot_metric_heatmap_grid(
    rand_heatmap_data,
    title='GMM Rand index heatmap across eps_times and top-neighbor percentages',
    save_path=rand_heatmap_path,
    annotate_cells=True,
)
plt.show()
print(f"Saved GMM Rand heatmap figure to {rand_heatmap_path}")

np.savez_compressed(
    rand_cache_path,
    eps_times=rand_eps_values,
    pct_values=rand_pct_values,
    eot_t=np.asarray(rand_eot_t, dtype=int),
    rand_cluster_seed=np.asarray(rand_cluster_seed, dtype=int),
    y_labels=np.asarray(rand_heatmap_data['y_labels']),
    gmm_rand_heatmap=rand_heatmap,
    gmm_rand_eot=rand_eot,
)
print(f"Saved GMM Rand heatmap matrices to {rand_cache_path}")


## GMM Theta Sweep at Fixed eps_times=10


In [ ]:
theta_values = [1.0, 1.5, 2.0, 2.5, 3.0]
theta_sweep_eps_times = 10.0
expected_tsne_seeds = [0, 1, 2, 3]
rand_cluster_seed = 0

if int(eot_t) != 3:
    raise ValueError(f'Expected eot_t=0 for this theta sweep, got {eot_t}')
if [int(seed) for seed in tsne_seeds] != expected_tsne_seeds:
    raise ValueError(
        f'Expected tsne_seeds={expected_tsne_seeds} for this theta sweep, got {tsne_seeds}'
    )

if 'cache_dir' in globals():
    theta_cache_dir = Path(cache_dir)
else:
    theta_cache_dir = Path('../data/benchmarking')
theta_cache_dir.mkdir(parents=True, exist_ok=True)

theta_figure_dir = theta_cache_dir / 'figures'
theta_figure_dir.mkdir(parents=True, exist_ok=True)

def theta_to_tag(theta):
    return f'{theta:g}'.replace('-', 'm').replace('.', 'p')

theta_benchmark_results = {}
theta_eot_references = {}
theta_rand_summaries = {}
theta_eot_rand = {}

theta_pct_values = None

theta_labels = [f'{theta:g}' for theta in theta_values]

for theta in theta_values:
    theta_tag = theta_to_tag(theta)
    theta_data_bundle = prepare_benchmark_data(
        seed=seed,
        n0_ring=n0_ring,
        n1_ring=n1_ring,
        gmm_n0=gmm_n0,
        gmm_n1=gmm_n1,
        gmm_p=gmm_p,
        gmm_q=gmm_q,
        gmm_theta=theta,
    )
    theta_plans = compute_ot_plans_for_eps(
        theta_data_bundle,
        eps_times=theta_sweep_eps_times,
        tol=tol,
        max_iter=max_iter,
    )

    theta_eot_cache_path = (
        theta_cache_dir
        / f'eot_reference_theta_sweep_eps_times_10_theta_{theta_tag}.npz'
    )
    theta_eot_reference = compute_eot_reference_from_plans(
        theta_data_bundle,
        theta_plans,
        t=eot_t,
        embd_dim=eot_embd_dim,
        k_conc=k_conc,
        k_cross=k_cross,
        cache_path=theta_eot_cache_path,
    )

    theta_block_matrices = build_four_block_matrices(theta_plans)
    theta_benchmark_cache_path = (
        theta_cache_dir
        / f'top_pct_neighbors_theta_sweep_eps_times_10_theta_{theta_tag}.npz'
    )
    theta_benchmark = run_top_pct_neighbors_benchmark(
        theta_data_bundle,
        theta_block_matrices,
        pct_values=pct_values,
        tsne_seeds=tsne_seeds,
        viz_seed=viz_seed,
        n_epochs=n_epochs,
        k_conc=k_conc,
        k_cross=k_cross,
        cache_path=theta_benchmark_cache_path,
    )

    theta_gt_clusters = np.concatenate(
        [theta_data_bundle['gmm']['clusters0'], theta_data_bundle['gmm']['clusters1']]
    )
    theta_rand_summary = compute_gmm_rand_summary(
        theta_benchmark,
        theta_gt_clusters,
        cluster_seed=rand_cluster_seed,
    )
    theta_direct_rand = compute_eot_gmm_rand(
        theta_eot_reference,
        theta_gt_clusters,
        cluster_seed=rand_cluster_seed,
    )

    theta_benchmark_results[float(theta)] = theta_benchmark
    theta_eot_references[float(theta)] = theta_eot_reference
    theta_rand_summaries[float(theta)] = theta_rand_summary
    theta_eot_rand[float(theta)] = float(theta_direct_rand)

    if theta_pct_values is None:
        theta_pct_values = np.asarray(theta_benchmark['pct_values'], dtype=float)

    gmm_metrics = theta_eot_reference['gmm']['metrics']
    print(
        f"theta={theta:g}: "
        f"EOT conc={gmm_metrics['concordance']:.4f}, "
        f"EOT xconc={gmm_metrics['cross_concordance']:.4f}, "
        f"EOT rand={theta_direct_rand:.4f}"
    )

concordance_theta_heatmap = build_metric_heatmap_data(
    theta_benchmark_results,
    theta_eot_references,
    metric_name='concordance',
    dataset_names=('gmm',),
)
concordance_theta_heatmap['y_labels'] = theta_labels
concordance_theta_heatmap['eps_times'] = np.asarray(theta_values, dtype=float)

cross_concordance_theta_heatmap = build_metric_heatmap_data(
    theta_benchmark_results,
    theta_eot_references,
    metric_name='cross_concordance',
    dataset_names=('gmm',),
)
cross_concordance_theta_heatmap['y_labels'] = theta_labels
cross_concordance_theta_heatmap['eps_times'] = np.asarray(theta_values, dtype=float)

rand_top_pct_matrix = np.stack(
    [theta_rand_summaries[float(theta)]['mean'] for theta in theta_values],
    axis=0,
)
rand_eot_column = np.asarray(
    [theta_eot_rand[float(theta)] for theta in theta_values],
    dtype=float,
)
rand_heatmap_data = {
    'metric_name': 'rand',
    'metric_label': 'Rand Index',
    'eps_times': np.asarray(theta_values, dtype=float),
    'pct_values': np.asarray(theta_pct_values, dtype=float),
    'eot_t': int(eot_t),
    'dataset_names': ('gmm',),
    'datasets': {
        'gmm': {
            'top_pct_matrix': rand_top_pct_matrix,
            'eot_column': rand_eot_column,
        }
    },
    'y_labels': theta_labels,
    'top_pct_column_count': int(np.asarray(theta_pct_values, dtype=float).size),
}

gmm_concordance_theta_heatmap_path = (
    theta_figure_dir / 'gmm_concordance_theta_pct_heatmap_eps_times_10.png'
)
fig, axes = plot_metric_heatmap_grid(
    concordance_theta_heatmap,
    title='GMM concordance heatmap across theta and top-neighbor percentages (eps_times=10)',
    annotate_cells=True,
)
axes['gmm']['main'].set_ylabel('theta')
fig.savefig(gmm_concordance_theta_heatmap_path, bbox_inches='tight')
plt.show()
print(f'Saved GMM concordance theta heatmap figure to {gmm_concordance_theta_heatmap_path}')

gmm_cross_concordance_theta_heatmap_path = (
    theta_figure_dir / 'gmm_cross_concordance_theta_pct_heatmap_eps_times_10.png'
)
fig, axes = plot_metric_heatmap_grid(
    cross_concordance_theta_heatmap,
    title='GMM cross-concordance heatmap across theta and top-neighbor percentages (eps_times=10)',
    annotate_cells=True,
)
axes['gmm']['main'].set_ylabel('theta')
fig.savefig(gmm_cross_concordance_theta_heatmap_path, bbox_inches='tight')
plt.show()
print(
    'Saved GMM cross-concordance theta heatmap figure to '
    f'{gmm_cross_concordance_theta_heatmap_path}'
)

gmm_rand_theta_heatmap_path = (
    theta_figure_dir / 'gmm_rand_theta_pct_heatmap_eps_times_10.png'
)
fig, axes = plot_metric_heatmap_grid(
    rand_heatmap_data,
    title='GMM Rand index heatmap across theta and top-neighbor percentages (eps_times=10)',
    annotate_cells=True,
)
axes['gmm']['main'].set_ylabel('theta')
fig.savefig(gmm_rand_theta_heatmap_path, bbox_inches='tight')
plt.show()
print(f'Saved GMM Rand theta heatmap figure to {gmm_rand_theta_heatmap_path}')

theta_heatmap_cache_path = theta_cache_dir / 'gmm_theta_pct_heatmaps_eps_times_10.npz'
np.savez_compressed(
    theta_heatmap_cache_path,
    theta_values=np.asarray(theta_values, dtype=float),
    pct_values=np.asarray(theta_pct_values, dtype=float),
    tsne_seeds=np.asarray(tsne_seeds, dtype=int),
    eot_t=np.asarray(eot_t, dtype=int),
    eps_times_fixed=np.asarray(theta_sweep_eps_times, dtype=float),
    y_labels=np.asarray(theta_labels),
    concordance_top_pct_matrix=np.asarray(
        concordance_theta_heatmap['datasets']['gmm']['top_pct_matrix'],
        dtype=float,
    ),
    concordance_eot_column=np.asarray(
        concordance_theta_heatmap['datasets']['gmm']['eot_column'],
        dtype=float,
    ),
    cross_concordance_top_pct_matrix=np.asarray(
        cross_concordance_theta_heatmap['datasets']['gmm']['top_pct_matrix'],
        dtype=float,
    ),
    cross_concordance_eot_column=np.asarray(
        cross_concordance_theta_heatmap['datasets']['gmm']['eot_column'],
        dtype=float,
    ),
    rand_top_pct_matrix=np.asarray(rand_top_pct_matrix, dtype=float),
    rand_eot_column=np.asarray(rand_eot_column, dtype=float),
)
print(f'Saved GMM theta heatmap matrices to {theta_heatmap_cache_path}')


In [ ]:
# Recompute the GMM theta-sweep Rand heatmap from cached embeddings/checkpoints.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

for candidate_src in (Path('../src'), Path('src')):
    resolved_src = candidate_src.resolve()
    if candidate_src.exists() and str(resolved_src) not in sys.path:
        sys.path.insert(0, str(resolved_src))

if any(
    name not in globals()
    for name in ('compute_eot_gmm_rand', 'compute_gmm_rand_summary', 'plot_metric_heatmap_grid')
):
    from ne_aligned_utils.benchmarking import (
        compute_eot_gmm_rand,
        compute_gmm_rand_summary,
        plot_metric_heatmap_grid,
    )

if 'make_joint_gmm' not in globals():
    from ne_aligned_utils.data import make_joint_gmm

theta_values_for_rand_viz = [
    float(theta)
    for theta in globals().get('theta_values', [1.0, 1.5, 2.0, 2.5, 3.0])
]
theta_sweep_eps_times_for_rand_viz = float(
    globals().get('theta_sweep_eps_times', 10.0)
)
rand_cluster_seed_for_viz = int(globals().get('rand_cluster_seed', 0))
gmm_q_for_rand_viz = int(globals().get('gmm_q', 6))

def format_eps_token_local(eps_times):
    return f'{float(eps_times):g}'.replace('-', 'm').replace('.', 'p')

def theta_to_tag_local(theta):
    return f'{float(theta):g}'.replace('-', 'm').replace('.', 'p')

def build_cluster_lookup(clean, clusters):
    return {
        np.ascontiguousarray(row).tobytes(): int(cluster)
        for row, cluster in zip(np.asarray(clean), np.asarray(clusters))
    }

def align_clusters_to_cached_clean(cached_clean, regenerated_clean, regenerated_clusters, theta, data_seed):
    cluster_lookup = build_cluster_lookup(regenerated_clean, regenerated_clusters)
    try:
        return np.asarray(
            [
                cluster_lookup[np.ascontiguousarray(row).tobytes()]
                for row in np.asarray(cached_clean)
            ],
            dtype=int,
        )
    except KeyError as exc:
        raise KeyError(
            'Could not align cached GMM clean samples with regenerated clusters '
            f'for theta={theta:g}, data_seed={data_seed}.'
        ) from exc

if 'cache_dir' in globals():
    theta_rand_cache_dir = Path(cache_dir)
else:
    theta_rand_cache_candidates = [Path('../data/benchmarking'), Path('data/benchmarking')]
    theta_rand_cache_dir = next(
        (
            path
            for path in theta_rand_cache_candidates
            if list(path.glob('top_pct_neighbors_theta_sweep_eps_times_*_theta_*.npz'))
            and list(path.glob('eot_reference_theta_sweep_eps_times_*_theta_*.npz'))
        ),
        theta_rand_cache_candidates[0],
    )

theta_eps_token = format_eps_token_local(theta_sweep_eps_times_for_rand_viz)
theta_rand_top_pct_rows = []
theta_rand_eot_values = []
theta_pct_values_for_rand = None

for theta in theta_values_for_rand_viz:
    theta_tag = theta_to_tag_local(theta)
    theta_benchmark_cache_path = (
        theta_rand_cache_dir
        / f'top_pct_neighbors_theta_sweep_eps_times_{theta_eps_token}_theta_{theta_tag}.npz'
    )
    theta_eot_cache_path = (
        theta_rand_cache_dir
        / f'eot_reference_theta_sweep_eps_times_{theta_eps_token}_theta_{theta_tag}.npz'
    )

    if not theta_benchmark_cache_path.exists():
        raise FileNotFoundError(
            f'Missing cached top-pct benchmark for theta={theta:g}: {theta_benchmark_cache_path}'
        )
    if not theta_eot_cache_path.exists():
        raise FileNotFoundError(
            f'Missing cached EOT reference for theta={theta:g}: {theta_eot_cache_path}'
        )

    with np.load(theta_benchmark_cache_path) as benchmark_npz:
        pct_values_loaded = np.asarray(benchmark_npz['pct_values'], dtype=float)
        tsne_seeds_loaded = [int(seed) for seed in benchmark_npz['tsne_seeds']]
        benchmark_clean0 = np.asarray(benchmark_npz['gmm_clean0'])
        benchmark_clean1 = np.asarray(benchmark_npz['gmm_clean1'])
        data_seed_loaded = int(benchmark_npz['data_seed'])

        benchmark_results = {
            'eps_times': float(benchmark_npz['eps_times']),
            'pct_values': [float(pct) for pct in pct_values_loaded],
            'tsne_seeds': tsne_seeds_loaded,
            'results': {},
        }
        for row_idx, pct in enumerate(pct_values_loaded):
            benchmark_results['results'][float(pct)] = {
                'gmm_embeddings_by_seed': {
                    int(seed): np.asarray(benchmark_npz['gmm_embeddings'][row_idx, seed_idx])
                    for seed_idx, seed in enumerate(tsne_seeds_loaded)
                }
            }

    with np.load(theta_eot_cache_path) as eot_npz:
        eot_reference = {
            't': int(eot_npz['t']),
            'gmm': {'embedding': np.asarray(eot_npz['gmm_embedding'])},
        }

    regenerated_clean0, regenerated_clean1, clusters0, clusters1 = make_joint_gmm(
        n0=int(benchmark_clean0.shape[0]),
        n1=int(benchmark_clean1.shape[0]),
        q=gmm_q_for_rand_viz,
        seed=data_seed_loaded,
        theta=float(theta),
        clean=True,
    )
    gt_clusters = np.concatenate([
        align_clusters_to_cached_clean(
            benchmark_clean0,
            regenerated_clean0,
            clusters0,
            theta=float(theta),
            data_seed=data_seed_loaded,
        ),
        align_clusters_to_cached_clean(
            benchmark_clean1,
            regenerated_clean1,
            clusters1,
            theta=float(theta),
            data_seed=data_seed_loaded,
        ),
    ])

    rand_summary = compute_gmm_rand_summary(
        benchmark_results,
        gt_clusters=gt_clusters,
        cluster_seed=rand_cluster_seed_for_viz,
    )
    eot_rand = compute_eot_gmm_rand(
        eot_reference,
        gt_clusters=gt_clusters,
        cluster_seed=rand_cluster_seed_for_viz,
    )

    if theta_pct_values_for_rand is None:
        theta_pct_values_for_rand = pct_values_loaded
    elif theta_pct_values_for_rand.shape != pct_values_loaded.shape or not np.allclose(
        theta_pct_values_for_rand,
        pct_values_loaded,
    ):
        raise ValueError('All cached theta benchmark files must share the same pct_values ordering.')

    theta_rand_top_pct_rows.append(np.asarray(rand_summary['mean'], dtype=float))
    theta_rand_eot_values.append(float(eot_rand))

recomputed_rand_top_pct_matrix = np.stack(theta_rand_top_pct_rows, axis=0)
recomputed_rand_eot_column = np.asarray(theta_rand_eot_values, dtype=float)

recomputed_rand_heatmap_data = {
    'metric_name': 'rand',
    'metric_label': 'Rand Index',
    'eps_times': np.asarray(theta_values_for_rand_viz, dtype=float),
    'pct_values': np.asarray(theta_pct_values_for_rand, dtype=float),
    'eot_t': int(eot_reference['t']),
    'dataset_names': ('gmm',),
    'datasets': {
        'gmm': {
            'top_pct_matrix': recomputed_rand_top_pct_matrix,
            'eot_column': recomputed_rand_eot_column,
        }
    },
    'y_labels': [f'{theta:g}' for theta in theta_values_for_rand_viz],
    'top_pct_column_count': int(np.asarray(theta_pct_values_for_rand, dtype=float).size),
}

fig, axes = plot_metric_heatmap_grid(
    recomputed_rand_heatmap_data,
    title=(
        'GMM Rand index heatmap across theta and top-neighbor percentages '
        f'(eps_times={theta_sweep_eps_times_for_rand_viz:g})'
    ),
    annotate_cells=True,
)
axes['gmm']['main'].set_ylabel('theta')
plt.show()

theta_heatmap_cache_path = (
    theta_rand_cache_dir / f'gmm_theta_pct_heatmaps_eps_times_{theta_eps_token}.npz'
)
if theta_heatmap_cache_path.exists():
    with np.load(theta_heatmap_cache_path) as theta_heatmap_npz:
        saved_rand_top_pct_matrix = np.asarray(
            theta_heatmap_npz['rand_top_pct_matrix'],
            dtype=float,
        )
        saved_rand_eot_column = np.asarray(
            theta_heatmap_npz['rand_eot_column'],
            dtype=float,
        )
    print(
        'Recomputed Rand index from cached embeddings. '
        f'Max abs delta vs saved heatmap cache: '
        f'top-pct={np.max(np.abs(recomputed_rand_top_pct_matrix - saved_rand_top_pct_matrix)):.6g}, '
        f'reference={np.max(np.abs(recomputed_rand_eot_column - saved_rand_eot_column)):.6g}'
    )
else:
    print(f'Recomputed Rand index from cached embeddings in {theta_rand_cache_dir}')


### EOT Embedding Check

These cells reuse the cached theta-sweep EOT outputs in `data/benchmarking/`. The `eot_reference_theta_sweep_eps_times_10_theta_*.npz` files store the saved embeddings and concordance/cross-concordance/FOSCTTM metrics, and `gmm_theta_pct_heatmaps_eps_times_10.npz` stores the saved EOT Rand indices. If those files are present, you do not need to rerun the EOT computation.

In [ ]:
# Load cached GMM EOT theta-sweep embeddings and metrics without recomputing them.
import pandas as pd

theta_values_for_viz = [float(theta) for theta in globals().get('theta_values', [1.0, 1.5, 2.0, 2.5, 3.0])]
theta_sweep_eps_times_for_viz = float(globals().get('theta_sweep_eps_times', 10.0))

if 'cache_dir' in globals():
    theta_viz_cache_dir = Path(cache_dir)
else:
    theta_viz_cache_candidates = [Path('../data/benchmarking'), Path('data/benchmarking')]
    theta_viz_cache_dir = next(
        (
            path
            for path in theta_viz_cache_candidates
            if (path / 'gmm_theta_pct_heatmaps_eps_times_10.npz').exists()
        ),
        theta_viz_cache_candidates[0],
    )

def theta_to_tag(theta):
    return f'{theta:g}'.replace('-', 'm').replace('.', 'p')

theta_rand_by_value = {}
theta_heatmap_cache_path = theta_viz_cache_dir / 'gmm_theta_pct_heatmaps_eps_times_10.npz'
if theta_heatmap_cache_path.exists():
    with np.load(theta_heatmap_cache_path) as theta_heatmap_npz:
        theta_rand_by_value = {
            float(theta): float(rand)
            for theta, rand in zip(
                theta_heatmap_npz['theta_values'],
                theta_heatmap_npz['rand_eot_column'],
            )
        }

theta_eot_gmm_by_theta = {}
theta_eot_metric_rows = []

for theta in theta_values_for_viz:
    theta_tag = theta_to_tag(theta)
    theta_eot_cache_path = (
        theta_viz_cache_dir
        / f'eot_reference_theta_sweep_eps_times_10_theta_{theta_tag}.npz'
    )
    if not theta_eot_cache_path.exists():
        raise FileNotFoundError(
            f'Missing cached EOT reference for theta={theta:g}: {theta_eot_cache_path}'
        )

    with np.load(theta_eot_cache_path) as eot_npz:
        metric_names_in_cache = [str(name) for name in eot_npz['metric_names']]
        metric_idx = {name: idx for idx, name in enumerate(metric_names_in_cache)}
        gmm_metrics = {
            name: float(eot_npz['gmm_metrics'][metric_idx[name]])
            for name in metric_names_in_cache
        }
        gmm_embedding = np.asarray(eot_npz['gmm_embedding'])
        n0 = int(eot_npz['gmm_clean0'].shape[0])
        n1 = int(eot_npz['gmm_clean1'].shape[0])

        theta_eot_gmm_by_theta[float(theta)] = {
            'embedding': gmm_embedding,
            'n0': n0,
            'n1': n1,
            't': int(eot_npz['t']),
            'data_seed': int(eot_npz['data_seed']),
            'cache_path': str(theta_eot_cache_path),
            'metrics': gmm_metrics,
        }

    theta_eot_metric_rows.append(
        {
            'theta': float(theta),
            'eps_times': theta_sweep_eps_times_for_viz,
            't': theta_eot_gmm_by_theta[float(theta)]['t'],
            'data_seed': theta_eot_gmm_by_theta[float(theta)]['data_seed'],
            'concordance': gmm_metrics['concordance'],
            'cross_concordance': gmm_metrics['cross_concordance'],
            'foscttm': gmm_metrics['foscttm'],
            'rand': theta_rand_by_value.get(float(theta), np.nan),
            'cache_path': str(theta_eot_cache_path),
        }
    )

theta_eot_metrics_df = pd.DataFrame(theta_eot_metric_rows).sort_values('theta').reset_index(drop=True)
print(f'Loaded cached theta-sweep EOT references from {theta_viz_cache_dir}')


In [ ]:
# Visualize the cached GMM EOT embeddings across theta.
# Each panel is recentered independently, but all panels share one
# global scalar display scale and the same axis limits.
from matplotlib.lines import Line2D

def center_embedding_for_display(embedding):
    return embedding - embedding.mean(axis=0, keepdims=True)

centered_embeddings_by_theta = {
    float(theta): center_embedding_for_display(
        theta_eot_gmm_by_theta[float(theta)]['embedding']
    )
    for theta in theta_values_for_viz
}
all_centered_embeddings = np.concatenate(
    [centered_embeddings_by_theta[float(theta)] for theta in theta_values_for_viz],
    axis=0,
)
global_scale = max(float(np.max(np.std(all_centered_embeddings, axis=0, ddof=1))), 1e-12)
scaled_all_embeddings = all_centered_embeddings / global_scale
global_q_lo = np.quantile(scaled_all_embeddings, 0.01, axis=0)
global_q_hi = np.quantile(scaled_all_embeddings, 0.99, axis=0)
global_pad = 0.08 * np.maximum(global_q_hi - global_q_lo, 1e-6)

fig, axes = plt.subplots(
    1,
    len(theta_values_for_viz),
    figsize=(3.2 * len(theta_values_for_viz), 4.1),
    squeeze=False,
)

for ax, theta in zip(axes.ravel(), theta_values_for_viz):
    theta_reference = theta_eot_gmm_by_theta[float(theta)]
    n0 = theta_reference['n0']
    viz_embedding = centered_embeddings_by_theta[float(theta)] / global_scale
    batch0 = viz_embedding[:n0]
    batch1 = viz_embedding[n0:]

    ax.scatter(
        batch0[:, 0],
        batch0[:, 1],
        s=6,
        c='tab:red',
        alpha=0.45,
        linewidths=0,
    )
    ax.scatter(
        batch1[:, 0],
        batch1[:, 1],
        s=6,
        c='tab:blue',
        alpha=0.45,
        linewidths=0,
    )
    ax.set_title(f'theta={theta:g}')
    ax.set_aspect('equal')
    ax.set_xlim(global_q_lo[0] - global_pad[0], global_q_hi[0] + global_pad[0])
    ax.set_ylim(global_q_lo[1] - global_pad[1], global_q_hi[1] + global_pad[1])
    ax.axis('off')

legend_handles = [
    Line2D([], [], marker='o', linestyle='', color='tab:red', markersize=6, label='batch 0'),
    Line2D([], [], marker='o', linestyle='', color='tab:blue', markersize=6, label='batch 1'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.03))
fig.suptitle(
    f'Centered GMM EOT embeddings (eps_times={theta_sweep_eps_times_for_viz:g})',
    y=0.98,
)


fig.tight_layout(rect=[0, 0.12, 1, 0.92])
plt.show()


In [ ]:
# Show the cached EOT metrics underneath the embedding plots.
from IPython.display import HTML, Markdown, display

display(HTML("<div style='height: 28px;'></div>"))
display(Markdown("#### Cached GMM EOT metrics"))

metrics_table = theta_eot_metrics_df[
    ['theta', 't', 'data_seed', 'concordance', 'cross_concordance', 'foscttm', 'rand']
].rename(
    columns={
        'theta': 'theta',
        't': 'EOT t',
        'data_seed': 'data seed',
        'concordance': 'concordance',
        'cross_concordance': 'cross-concordance',
        'foscttm': 'FOSCTTM',
        'rand': 'Rand index',
    }
)

display(
    metrics_table.style.hide(axis='index').format(
        {
            'theta': '{:.1f}',
            'EOT t': '{:d}',
            'data seed': '{:d}',
            'concordance': '{:.4f}',
            'cross-concordance': '{:.4f}',
            'FOSCTTM': '{:.4f}',
            'Rand index': '{:.4f}',
        }
    ).set_table_styles([
        {
            'selector': 'table',
            'props': [('margin-top', '8px')],
        }
    ])
)


## GMM EOT Grid Across eps_times and theta

The cells below reuse the existing EOT caches when they are available, and only compute the missing `eps_times × theta` combinations once. The embedding grid and all three heatmaps then reuse the same loaded results without recalculating them.

In [ ]:
# Load or compute the GMM EOT reference grid across eps_times and theta once,
# then reuse it for the embedding grid and metric heatmaps below.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

for candidate_src in (Path('../src'), Path('src')):
    resolved_src = candidate_src.resolve()
    if candidate_src.exists() and str(resolved_src) not in sys.path:
        sys.path.insert(0, str(resolved_src))

if any(
    name not in globals()
    for name in (
        'prepare_benchmark_data',
        'compute_ot_plans_for_eps',
        'compute_eot_reference_from_plans',
        'compute_eot_gmm_rand',
    )
):
    from ne_aligned_utils.benchmarking import (
        compute_eot_gmm_rand,
        compute_eot_reference_from_plans,
        compute_ot_plans_for_eps,
        prepare_benchmark_data,
    )

if 'make_joint_gmm' not in globals():
    from ne_aligned_utils.data import make_joint_gmm

grid_eps_values = [1.0, 10.0, 100.0, 300.0]
grid_theta_values = [1.0, 1.5, 2.0, 2.5, 3.0]
grid_metric_names = ('concordance', 'cross_concordance', 'foscttm', 'rand')
grid_metric_labels = {
    'concordance': 'Concordance',
    'cross_concordance': 'Cross-concordance',
    'foscttm': 'FOSCTTM',
    'rand': 'Rand index',
}

grid_seed = int(globals().get('seed', 0))
grid_n0_ring = int(globals().get('n0_ring', 1000))
grid_n1_ring = int(globals().get('n1_ring', 5000))
grid_gmm_n0 = int(globals().get('gmm_n0', 600))
grid_gmm_n1 = int(globals().get('gmm_n1', 600))
grid_gmm_p = int(globals().get('gmm_p', 1000))
grid_gmm_q = int(globals().get('gmm_q', 6))
grid_base_theta = float(globals().get('gmm_theta', 3.0))
grid_k_conc = int(globals().get('k_conc', 50))
grid_k_cross = int(globals().get('k_cross', 50))
grid_eot_t = int(globals().get('eot_t', 0))
grid_eot_embd_dim = int(globals().get('eot_embd_dim', 2))
grid_tol = float(globals().get('tol', 1e-12))
grid_max_iter = int(globals().get('max_iter', int(1e4)))
grid_rand_cluster_seed = int(globals().get('rand_cluster_seed', 0))

if 'cache_dir' in globals():
    eps_theta_grid_cache_dir = Path(cache_dir)
else:
    eps_theta_grid_candidates = [Path('../data/benchmarking'), Path('data/benchmarking')]
    eps_theta_grid_cache_dir = next(
        (
            path
            for path in eps_theta_grid_candidates
            if (path / 'eot_reference_eps_times_10.npz').exists()
            or list(path.glob('eot_reference_theta_sweep_eps_times_10_theta_*.npz'))
        ),
        eps_theta_grid_candidates[0],
    )
eps_theta_grid_cache_dir.mkdir(parents=True, exist_ok=True)

def format_eps_token_local(eps_times):
    return f'{float(eps_times):g}'.replace('-', 'm').replace('.', 'p')

def theta_to_tag_local(theta):
    return f'{float(theta):g}'.replace('-', 'm').replace('.', 'p')

def resolve_gmm_eot_combo_cache_paths(cache_dir, eps_times, theta, base_theta):
    eps = float(eps_times)
    theta = float(theta)
    combo_cache_path = cache_dir / (
        f'eot_reference_eps_times_{format_eps_token_local(eps)}_theta_{theta_to_tag_local(theta)}.npz'
    )
    candidates = []
    if eps == 10.0:
        candidates.append(
            cache_dir / f'eot_reference_theta_sweep_eps_times_10_theta_{theta_to_tag_local(theta)}.npz'
        )
    if theta == float(base_theta):
        candidates.append(cache_dir / f'eot_reference_eps_times_{format_eps_token_local(eps)}.npz')
    candidates.append(combo_cache_path)
    existing_cache_path = next((path for path in candidates if path.exists()), combo_cache_path)
    return existing_cache_path, combo_cache_path

def load_gmm_eot_reference_cache(cache_path):
    with np.load(cache_path) as eot_npz:
        metric_names_in_cache = [str(name) for name in eot_npz['metric_names']]
        metric_idx = {name: idx for idx, name in enumerate(metric_names_in_cache)}
        return {
            'embedding': np.asarray(eot_npz['gmm_embedding']),
            'metrics': {
                name: float(eot_npz['gmm_metrics'][metric_idx[name]])
                for name in metric_names_in_cache
            },
            'n0': int(eot_npz['gmm_clean0'].shape[0]),
            'n1': int(eot_npz['gmm_clean1'].shape[0]),
            't': int(eot_npz['t']),
            'data_seed': int(eot_npz['data_seed']),
            'cache_path': str(cache_path),
        }

def load_sorted_gmm_gt_clusters_for_theta(seed, n0, n1, p, q, theta):
    _, _, clusters0, clusters1 = make_joint_gmm(
        n0=n0,
        n1=n1,
        p=p,
        q=q,
        theta=theta,
        seed=seed,
        clean=True,
    )
    sort_idx0 = np.argsort(clusters0)
    sort_idx1 = np.argsort(clusters1)
    return np.concatenate([
        np.asarray(clusters0[sort_idx0]),
        np.asarray(clusters1[sort_idx1]),
    ])

theta_data_bundles_by_theta = {}
gmm_gt_clusters_by_theta = {
    float(theta): load_sorted_gmm_gt_clusters_for_theta(
        seed=grid_seed,
        n0=grid_gmm_n0,
        n1=grid_gmm_n1,
        p=grid_gmm_p,
        q=grid_gmm_q,
        theta=theta,
    )
    for theta in grid_theta_values
}
eps_theta_gmm_eot_grid = {}
gmm_eot_metric_matrices = {
    metric_name: np.empty((len(grid_eps_values), len(grid_theta_values)), dtype=float)
    for metric_name in grid_metric_names
}
gmm_eot_embedding_grid = None
gmm_eot_cache_path_grid = np.empty((len(grid_eps_values), len(grid_theta_values)), dtype=object)
loaded_combo_keys = []
computed_combo_keys = []

for eps_idx, eps_times in enumerate(grid_eps_values):
    for theta_idx, theta in enumerate(grid_theta_values):
        existing_cache_path, combo_cache_path = resolve_gmm_eot_combo_cache_paths(
            eps_theta_grid_cache_dir,
            eps_times=eps_times,
            theta=theta,
            base_theta=grid_base_theta,
        )

        if existing_cache_path.exists():
            gmm_eot_reference = load_gmm_eot_reference_cache(existing_cache_path)
            loaded_combo_keys.append((float(eps_times), float(theta), existing_cache_path.name))
        else:
            if float(theta) not in theta_data_bundles_by_theta:
                theta_data_bundles_by_theta[float(theta)] = prepare_benchmark_data(
                    seed=grid_seed,
                    n0_ring=grid_n0_ring,
                    n1_ring=grid_n1_ring,
                    gmm_n0=grid_gmm_n0,
                    gmm_n1=grid_gmm_n1,
                    gmm_p=grid_gmm_p,
                    gmm_q=grid_gmm_q,
                    gmm_theta=theta,
                )

            theta_data_bundle = theta_data_bundles_by_theta[float(theta)]
            theta_plans = compute_ot_plans_for_eps(
                theta_data_bundle,
                eps_times=eps_times,
                tol=grid_tol,
                max_iter=grid_max_iter,
            )
            theta_eot_reference = compute_eot_reference_from_plans(
                theta_data_bundle,
                theta_plans,
                t=grid_eot_t,
                embd_dim=grid_eot_embd_dim,
                k_conc=grid_k_conc,
                k_cross=grid_k_cross,
                cache_path=combo_cache_path,
            )
            gmm_eot_reference = {
                'embedding': np.asarray(theta_eot_reference['gmm']['embedding']),
                'metrics': {
                    name: float(value)
                    for name, value in theta_eot_reference['gmm']['metrics'].items()
                },
                'n0': int(theta_data_bundle['gmm']['clean0'].shape[0]),
                'n1': int(theta_data_bundle['gmm']['clean1'].shape[0]),
                't': int(theta_eot_reference['t']),
                'data_seed': int(theta_data_bundle['seed']),
                'cache_path': str(combo_cache_path),
            }
            computed_combo_keys.append((float(eps_times), float(theta), combo_cache_path.name))

        if gmm_eot_embedding_grid is None:
            embedding_shape = tuple(gmm_eot_reference['embedding'].shape)
            gmm_eot_embedding_grid = np.empty(
                (len(grid_eps_values), len(grid_theta_values), *embedding_shape),
                dtype=float,
            )
            gmm_eot_n0 = int(gmm_eot_reference['n0'])
            gmm_eot_n1 = int(gmm_eot_reference['n1'])

        if int(gmm_eot_reference['t']) != int(grid_eot_t):
            raise ValueError(
                f"Expected EOT t={grid_eot_t}, got {gmm_eot_reference['t']} for eps_times={eps_times:g}, theta={theta:g}"
            )

        gmm_eot_reference['metrics']['rand'] = compute_eot_gmm_rand(
            {'gmm': {'embedding': gmm_eot_reference['embedding']}},
            gmm_gt_clusters_by_theta[float(theta)],
            cluster_seed=grid_rand_cluster_seed,
        )

        eps_theta_gmm_eot_grid[(float(eps_times), float(theta))] = gmm_eot_reference
        gmm_eot_embedding_grid[eps_idx, theta_idx] = gmm_eot_reference['embedding']
        gmm_eot_cache_path_grid[eps_idx, theta_idx] = gmm_eot_reference['cache_path']
        for metric_name in grid_metric_names:
            gmm_eot_metric_matrices[metric_name][eps_idx, theta_idx] = gmm_eot_reference['metrics'][metric_name]

def plot_gmm_eps_theta_embedding_grid(embedding_grid, n0, eps_values, theta_values, title=None):
    centered_grid = embedding_grid - embedding_grid.mean(axis=2, keepdims=True)
    row_display_params = []
    for eps_idx in range(len(eps_values)):
        row_centered_embeddings = centered_grid[eps_idx].reshape(-1, embedding_grid.shape[-1])
        row_scale = max(float(np.max(np.std(row_centered_embeddings, axis=0, ddof=1))), 1e-12)
        scaled_row_embeddings = row_centered_embeddings / row_scale
        row_q_lo = np.quantile(scaled_row_embeddings, 0.01, axis=0)
        row_q_hi = np.quantile(scaled_row_embeddings, 0.99, axis=0)
        row_pad = 0.08 * np.maximum(row_q_hi - row_q_lo, 1e-6)
        row_display_params.append((row_scale, row_q_lo, row_q_hi, row_pad))

    fig, axes = plt.subplots(
        len(eps_values),
        len(theta_values),
        figsize=(3.0 * len(theta_values), 2.8 * len(eps_values)),
        squeeze=False,
    )

    for eps_idx, eps_times in enumerate(eps_values):
        row_scale, row_q_lo, row_q_hi, row_pad = row_display_params[eps_idx]
        for theta_idx, theta in enumerate(theta_values):
            ax = axes[eps_idx, theta_idx]
            viz_embedding = centered_grid[eps_idx, theta_idx] / row_scale
            batch0 = viz_embedding[:n0]
            batch1 = viz_embedding[n0:]

            ax.scatter(
                batch0[:, 0],
                batch0[:, 1],
                s=6,
                c='tab:red',
                alpha=0.45,
                linewidths=0,
            )
            ax.scatter(
                batch1[:, 0],
                batch1[:, 1],
                s=6,
                c='tab:blue',
                alpha=0.45,
                linewidths=0,
            )

            if eps_idx == 0:
                ax.set_title(f'theta={theta:g}', fontsize=13)
            if theta_idx == 0:
                ax.text(
                    -0.18,
                    0.5,
                    f'eps={eps_times:g}',
                    transform=ax.transAxes,
                    rotation=90,
                    va='center',
                    ha='center',
                    fontsize=11,
                )

            ax.set_aspect('equal')
            ax.set_xlim(row_q_lo[0] - row_pad[0], row_q_hi[0] + row_pad[0])
            ax.set_ylim(row_q_lo[1] - row_pad[1], row_q_hi[1] + row_pad[1])
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

    legend_handles = [
        Line2D([], [], marker='o', linestyle='', color='tab:red', markersize=6, label='batch 0'),
        Line2D([], [], marker='o', linestyle='', color='tab:blue', markersize=6, label='batch 1'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.03))
    fig.suptitle(
        title or 'Centered GMM EOT embeddings across eps_times and theta',
        y=0.98,
        fontsize=15,
    )
    fig.text(
        0.5,
        0.06,
        'Each panel is recentered, and each eps_times row shares its own scalar display scale and axis limits.',
        ha='center',
        fontsize=9,
    )
    fig.tight_layout(rect=[0.03, 0.09, 1, 0.95])
    return fig, axes

def plot_gmm_eps_theta_metric_heatmap(metric_name, metric_matrix, eps_values, theta_values, title=None, cmap='viridis'):
    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    image = ax.imshow(metric_matrix, aspect='auto', interpolation='nearest', cmap=cmap)
    ax.set_xticks(np.arange(len(theta_values)))
    ax.set_xticklabels([f'{theta:g}' for theta in theta_values])
    ax.set_yticks(np.arange(len(eps_values)))
    ax.set_yticklabels([f'{eps:g}' for eps in eps_values])
    ax.set_xlabel('theta')
    ax.set_ylabel('eps_times')
    ax.set_title(title or f'GMM EOT {grid_metric_labels[metric_name]} across eps_times and theta')
    ax.grid(False)

    vmin = float(np.min(metric_matrix))
    vmax = float(np.max(metric_matrix))
    value_span = max(vmax - vmin, 1e-12)
    for row_idx in range(metric_matrix.shape[0]):
        for col_idx in range(metric_matrix.shape[1]):
            value = float(metric_matrix[row_idx, col_idx])
            text_color = 'white' if (value - vmin) / value_span < 0.55 else '#1f1f1f'
            ax.text(
                col_idx,
                row_idx,
                f'{value:.3f}',
                ha='center',
                va='center',
                fontsize=9,
                color=text_color,
                fontweight='semibold',
            )

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label(grid_metric_labels[metric_name])
    fig.tight_layout()
    return fig, ax

print(f'Loaded {len(loaded_combo_keys)} cached eps_times/theta combination(s).')
if computed_combo_keys:
    print(f'Computed and cached {len(computed_combo_keys)} missing combination(s):')
    for eps_times, theta, cache_name in computed_combo_keys:
        print(f'  eps_times={eps_times:g}, theta={theta:g} -> {cache_name}')
else:
    print('No new EOT references were needed.')

print(
    f"GMM EOT grid ready: embeddings={gmm_eot_embedding_grid.shape}, "
    f"eps_times={grid_eps_values}, theta={grid_theta_values}"
)


In [ ]:
fig, axes = plot_gmm_eps_theta_embedding_grid(
    gmm_eot_embedding_grid,
    n0=gmm_eot_n0,
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='Centered GMM EOT embeddings across eps_times and theta',
)
plt.show()


In [ ]:
from matplotlib.lines import Line2D

def plot_gmm_eps_theta_embedding_grid_by_cluster(
    embedding_grid,
    cluster_labels_by_theta,
    eps_values,
    theta_values,
    title=None,
):
    centered_grid = embedding_grid - embedding_grid.mean(axis=2, keepdims=True)
    row_display_params = []
    for eps_idx in range(len(eps_values)):
        row_centered_embeddings = centered_grid[eps_idx].reshape(-1, embedding_grid.shape[-1])
        row_scale = max(float(np.max(np.std(row_centered_embeddings, axis=0, ddof=1))), 1e-12)
        scaled_row_embeddings = row_centered_embeddings / row_scale
        row_q_lo = np.quantile(scaled_row_embeddings, 0.01, axis=0)
        row_q_hi = np.quantile(scaled_row_embeddings, 0.99, axis=0)
        row_pad = 0.08 * np.maximum(row_q_hi - row_q_lo, 1e-6)
        row_display_params.append((row_scale, row_q_lo, row_q_hi, row_pad))

    all_clusters = np.unique(
        np.concatenate([
            np.asarray(cluster_labels_by_theta[float(theta)])
            for theta in theta_values
        ])
    )
    cmap = plt.get_cmap('tab10', len(all_clusters))
    cluster_colors = {
        int(cluster): cmap(cluster_idx)
        for cluster_idx, cluster in enumerate(all_clusters)
    }

    fig, axes = plt.subplots(
        len(eps_values),
        len(theta_values),
        figsize=(3.0 * len(theta_values), 2.8 * len(eps_values)),
        squeeze=False,
    )

    for eps_idx, eps_times in enumerate(eps_values):
        row_scale, row_q_lo, row_q_hi, row_pad = row_display_params[eps_idx]
        for theta_idx, theta in enumerate(theta_values):
            ax = axes[eps_idx, theta_idx]
            viz_embedding = centered_grid[eps_idx, theta_idx] / row_scale
            cluster_labels = np.asarray(cluster_labels_by_theta[float(theta)])

            if cluster_labels.shape[0] != viz_embedding.shape[0]:
                raise ValueError(
                    f'Expected {viz_embedding.shape[0]} cluster labels for theta={theta:g}, got {cluster_labels.shape[0]}'
                )

            for cluster in all_clusters:
                cluster_mask = cluster_labels == cluster
                ax.scatter(
                    viz_embedding[cluster_mask, 0],
                    viz_embedding[cluster_mask, 1],
                    s=6,
                    c=[cluster_colors[int(cluster)]],
                    alpha=0.45,
                    linewidths=0,
                )

            if eps_idx == 0:
                ax.set_title(f'theta={theta:g}', fontsize=13)
            if theta_idx == 0:
                ax.text(
                    -0.18,
                    0.5,
                    f'eps={eps_times:g}',
                    transform=ax.transAxes,
                    rotation=90,
                    va='center',
                    ha='center',
                    fontsize=11,
                )

            ax.set_aspect('equal')
            ax.set_xlim(row_q_lo[0] - row_pad[0], row_q_hi[0] + row_pad[0])
            ax.set_ylim(row_q_lo[1] - row_pad[1], row_q_hi[1] + row_pad[1])
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

    legend_handles = [
        Line2D(
            [],
            [],
            marker='o',
            linestyle='',
            color=cluster_colors[int(cluster)],
            markersize=6,
            label=f'cluster {int(cluster)}',
        )
        for cluster in all_clusters
    ]
    fig.legend(
        handles=legend_handles,
        loc='lower center',
        ncol=len(all_clusters),
        frameon=False,
        bbox_to_anchor=(0.5, 0.03),
    )
    fig.suptitle(
        title or 'Centered GMM EOT embeddings across eps_times and theta (colored by original cluster)',
        y=0.98,
        fontsize=15,
    )
    fig.text(
        0.5,
        0.06,
        'Each panel is recentered, and each eps_times row shares its own scalar display scale and axis limits.',
        ha='center',
        fontsize=9,
    )
    fig.tight_layout(rect=[0.03, 0.09, 1, 0.95])
    return fig, axes

fig, axes = plot_gmm_eps_theta_embedding_grid_by_cluster(
    gmm_eot_embedding_grid,
    cluster_labels_by_theta=gmm_gt_clusters_by_theta,
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='Centered GMM EOT embeddings across eps_times and theta (colored by original cluster)',
)
plt.show()


### GMM EOT Metric Heatmaps

In [ ]:
fig, ax = plot_gmm_eps_theta_metric_heatmap(
    'concordance',
    gmm_eot_metric_matrices['concordance'],
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='GMM EOT concordance across eps_times and theta',
)
plt.show()


In [ ]:
fig, ax = plot_gmm_eps_theta_metric_heatmap(
    'cross_concordance',
    gmm_eot_metric_matrices['cross_concordance'],
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='GMM EOT cross-concordance across eps_times and theta',
)
plt.show()


In [ ]:
fig, ax = plot_gmm_eps_theta_metric_heatmap(
    'foscttm',
    gmm_eot_metric_matrices['foscttm'],
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='GMM EOT FOSCTTM across eps_times and theta',
    cmap='viridis_r',
)
plt.show()


In [ ]:
fig, ax = plot_gmm_eps_theta_metric_heatmap(
    'rand',
    gmm_eot_metric_matrices['rand'],
    eps_values=grid_eps_values,
    theta_values=grid_theta_values,
    title='GMM EOT Rand index across eps_times and theta',
    cmap='plasma',
)
plt.show()


In [ ]:
# Sweep the number of nontrivial singular vectors for the theta=1 GMM OT plans.
if any(name not in globals() for name in ('prepare_benchmark_data', 'compute_ot_plans_for_eps')):
    from ne_aligned_utils.benchmarking import (
        compute_ot_plans_for_eps,
        prepare_benchmark_data,
    )

if any(name not in globals() for name in ('compute_concordance', 'compute_cross_concordance', 'compute_kmeans_rand')):
    from ne_aligned_utils.metrics import (
        compute_concordance,
        compute_cross_concordance,
        compute_kmeans_rand,
    )

sv_theta = 1.0
sv_eps_values = [1.0, 10.0, 100.0, 300.0]
sv_seed = int(globals().get('grid_seed', globals().get('seed', 0)))
sv_n0_ring = int(globals().get('grid_n0_ring', globals().get('n0_ring', 1000)))
sv_n1_ring = int(globals().get('grid_n1_ring', globals().get('n1_ring', 5000)))
sv_gmm_n0 = int(globals().get('grid_gmm_n0', globals().get('gmm_n0', 600)))
sv_gmm_n1 = int(globals().get('grid_gmm_n1', globals().get('gmm_n1', 600)))
sv_gmm_p = int(globals().get('grid_gmm_p', globals().get('gmm_p', 1000)))
sv_gmm_q = int(globals().get('grid_gmm_q', globals().get('gmm_q', 6)))
sv_k_conc = int(globals().get('grid_k_conc', globals().get('k_conc', 50)))
sv_k_cross = int(globals().get('grid_k_cross', globals().get('k_cross', 50)))
sv_t = int(globals().get('grid_eot_t', globals().get('eot_t', 0)))
sv_tol = float(globals().get('grid_tol', globals().get('tol', 1e-12)))
sv_max_iter = int(globals().get('grid_max_iter', globals().get('max_iter', int(1e4))))
sv_rand_seed = int(globals().get('grid_rand_cluster_seed', globals().get('rand_cluster_seed', 0)))

sv_data_bundle = prepare_benchmark_data(
    seed=sv_seed,
    n0_ring=sv_n0_ring,
    n1_ring=sv_n1_ring,
    gmm_n0=sv_gmm_n0,
    gmm_n1=sv_gmm_n1,
    gmm_p=sv_gmm_p,
    gmm_q=sv_gmm_q,
    gmm_theta=sv_theta,
)
sv_clean0 = np.asarray(sv_data_bundle['gmm']['clean0'])
sv_clean1 = np.asarray(sv_data_bundle['gmm']['clean1'])
sv_clean = np.concatenate([sv_clean0, sv_clean1], axis=0)
sv_n0 = int(sv_clean0.shape[0])
sv_n1 = int(sv_clean1.shape[0])
sv_gt_clusters = np.concatenate([
    np.asarray(sv_data_bundle['gmm']['clusters0']),
    np.asarray(sv_data_bundle['gmm']['clusters1']),
])
sv_n_clusters = int(np.unique(sv_gt_clusters).size)
# Use the first 5 nontrivial singular-vector counts for the heatmap columns.
sv_counts = list(range(1, min(6, min(sv_n0, sv_n1))))

sv_metric_matrices = {
    'concordance': np.empty((len(sv_eps_values), len(sv_counts)), dtype=float),
    'cross_concordance': np.empty((len(sv_eps_values), len(sv_counts)), dtype=float),
    'rand': np.empty((len(sv_eps_values), len(sv_counts)), dtype=float),
}

for eps_idx, eps_times in enumerate(sv_eps_values):
    sv_plans = compute_ot_plans_for_eps(
        sv_data_bundle,
        eps_times=eps_times,
        tol=sv_tol,
        max_iter=sv_max_iter,
    )
    sv_plan = np.asarray(sv_plans['gmm']['P'], dtype=float)
    U, singular_values, Vt = np.linalg.svd(sv_plan, full_matrices=False)

    for count_idx, count in enumerate(sv_counts):
        weights = singular_values[1:1 + count] ** sv_t
        embd0 = U[:, 1:1 + count] * weights * np.sqrt(sv_n0)
        embd1 = Vt.T[:, 1:1 + count] * weights * np.sqrt(sv_n1)
        embd = np.concatenate([embd0, embd1], axis=0)

        sv_metric_matrices['concordance'][eps_idx, count_idx] = compute_concordance(
            sv_clean,
            embd,
            k=min(sv_k_conc, sv_clean.shape[0]),
        )
        sv_metric_matrices['cross_concordance'][eps_idx, count_idx] = compute_cross_concordance(
            sv_clean0,
            sv_clean1,
            embd0,
            embd1,
            k=min(sv_k_cross, sv_n0, sv_n1),
        )
        sv_metric_matrices['rand'][eps_idx, count_idx] = compute_kmeans_rand(
            embd,
            sv_gt_clusters,
            n_clusters=sv_n_clusters,
            seed=sv_rand_seed,
        )

gmm_theta1_sv_metric_matrices = sv_metric_matrices
gmm_theta1_sv_counts = sv_counts
gmm_theta1_sv_eps_values = sv_eps_values

sv_metric_labels = {
    'concordance': 'Concordance',
    'cross_concordance': 'X concordance',
    'rand': 'Rand index',
}
sv_cmaps = {
    'concordance': 'viridis',
    'cross_concordance': 'viridis',
    'rand': 'plasma',
}

fig, axes = plt.subplots(3, 1, figsize=(7.0, 12.5), squeeze=False)

for ax, metric_name in zip(axes.ravel(), ('concordance', 'cross_concordance', 'rand')):
    matrix = sv_metric_matrices[metric_name]
    image = ax.imshow(matrix, aspect='auto', interpolation='nearest', cmap=sv_cmaps[metric_name])
    ax.set_xticks(np.arange(len(sv_counts)))
    ax.set_xticklabels([str(count) for count in sv_counts])
    ax.set_yticks(np.arange(len(sv_eps_values)))
    ax.set_yticklabels([f'{eps:g}' for eps in sv_eps_values])
    ax.set_xlabel('nontrivial singular vectors')
    ax.set_ylabel('eps_times')
    ax.set_title(sv_metric_labels[metric_name])
    ax.grid(False)

    vmin = float(np.min(matrix))
    vmax = float(np.max(matrix))
    value_span = max(vmax - vmin, 1e-12)
    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            value = float(matrix[row_idx, col_idx])
            text_color = 'white' if (value - vmin) / value_span < 0.55 else '#1f1f1f'
            ax.text(
                col_idx,
                row_idx,
                f'{value:.3f}',
                ha='center',
                va='center',
                fontsize=9,
                color=text_color,
                fontweight='semibold',
            )

    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label(sv_metric_labels[metric_name])

fig.suptitle(
    f'GMM theta={sv_theta:g}: metrics vs added nontrivial singular vectors (t={sv_t})',
    y=1.03,
)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()



### Binarization

This section mirrors the `P -> A -> sparsify A -> Step 5 postprocessing -> t-SNE` pipeline: build the full 4-block matrix from the OT plan, sparsify that 4-block matrix with the same top-% rule as the main benchmark, then binarize, symmetrize, and normalize exactly as in Step 5 before running t-SNE.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ne_aligned_utils.benchmarking import (
    build_four_block_matrices,
    build_metric_heatmap_data,
    compute_eot_reference_from_plans,
    compute_ot_plans_for_eps,
    compute_shared_dataset_scale_bounds,
    format_eps_token,
    load_benchmark_cache,
    load_eot_reference_cache,
    make_eps_eot_cache_path,
    plot_gmm_pct_embedding_grid_by_cluster,
    plot_k50_concordance_summary,
    plot_metric_heatmap_grid,
    plot_pct_embedding_grid,
    postprocess_graph_matrix,
    prepare_benchmark_data,
    run_top_pct_neighbors_benchmark,
    sparsify_top_percentage_neighbors_graph,
)

seed = int(globals().get('seed', 0))
pct_values = list(globals().get('pct_values', [0.1, 0.5, 1, 5, 10, 12, 15, 17, 20, 22, 25, 30, 35, 40]))
tsne_seeds = list(globals().get('tsne_seeds', [0, 1, 2, 3]))
viz_seed = int(globals().get('viz_seed', 0))
n_epochs = int(globals().get('n_epochs', 200))
k_conc = int(globals().get('k_conc', 50))
k_cross = int(globals().get('k_cross', 50))
eot_t = int(globals().get('eot_t', 0))
eot_embd_dim = int(globals().get('eot_embd_dim', 2))
tol = float(globals().get('tol', 1e-12))
max_iter = int(globals().get('max_iter', int(1e4)))

n0_ring = int(globals().get('n0_ring', 1000))
n1_ring = int(globals().get('n1_ring', 5000))
gmm_n0 = int(globals().get('gmm_n0', 600))
gmm_n1 = int(globals().get('gmm_n1', 600))
gmm_p = int(globals().get('gmm_p', 1000))
gmm_q = int(globals().get('gmm_q', 6))
gmm_theta = float(globals().get('gmm_theta', 3.0))

cache_dir = Path(globals().get('cache_dir', Path('../data/benchmarking')))
cache_dir.mkdir(parents=True, exist_ok=True)

if viz_seed not in tsne_seeds:
    raise ValueError(f'viz_seed={viz_seed} must be included in tsne_seeds={tsne_seeds}')

if 'data_bundle' not in globals():
    data_bundle = prepare_benchmark_data(
        seed=seed,
        n0_ring=n0_ring,
        n1_ring=n1_ring,
        gmm_n0=gmm_n0,
        gmm_n1=gmm_n1,
        gmm_p=gmm_p,
        gmm_q=gmm_q,
        gmm_theta=gmm_theta,
    )

binarization_eps_values = [1.0, 10.0, 100.0, 300.0]

binarization_plan_name_by_eps = {
    1.0: 'plans_1',
    10.0: 'plans_10',
    100.0: 'plans_100',
    300.0: 'plans_1000',
}
binarization_block_name_by_eps = {
    1.0: 'block_matrices_1',
    10.0: 'block_matrices_10',
    100.0: 'block_matrices_100',
    300.0: 'block_matrices_1000',
}
binarization_eot_name_by_eps = {
    1.0: 'eot_reference_1',
    10.0: 'eot_reference_10',
    100.0: 'eot_reference_100',
    300.0: 'eot_reference_1000',
}


def make_binarization_cache_path(cache_dir, eps_times):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / (
        f'top_pct_neighbors_4block_step5_binarized_eps_times_{format_eps_token(eps_times)}.npz'
    )


def get_or_compute_binarization_plans(data_bundle, eps_times, tol, max_iter):
    plan_name = binarization_plan_name_by_eps.get(float(eps_times))
    if plan_name is not None and plan_name in globals():
        return globals()[plan_name]

    plans = compute_ot_plans_for_eps(
        data_bundle,
        eps_times=eps_times,
        tol=tol,
        max_iter=max_iter,
    )
    if plan_name is not None:
        globals()[plan_name] = plans
    return plans


def get_or_compute_binarization_block_matrices(plans, eps_times):
    block_name = binarization_block_name_by_eps.get(float(eps_times))
    if block_name is not None and block_name in globals():
        return globals()[block_name]

    block_matrices = build_four_block_matrices(plans)
    if block_name is not None:
        globals()[block_name] = block_matrices
    return block_matrices


def get_or_compute_binarization_eot_reference(
    data_bundle,
    plans,
    eps_times,
    cache_dir,
    t,
    embd_dim,
    k_conc,
    k_cross,
):
    ref_name = binarization_eot_name_by_eps.get(float(eps_times))
    if ref_name is not None and ref_name in globals():
        return globals()[ref_name]

    cache_path = make_eps_eot_cache_path(cache_dir, eps_times)
    if cache_path.exists():
        eot_reference = load_eot_reference_cache(cache_path)
    else:
        eot_reference = compute_eot_reference_from_plans(
            data_bundle,
            plans,
            t=t,
            embd_dim=embd_dim,
            k_conc=k_conc,
            k_cross=k_cross,
            cache_path=cache_path,
        )

    if ref_name is not None:
        globals()[ref_name] = eot_reference
    return eot_reference


In [ ]:
binarization_plans_by_eps = {}
binarization_block_matrices_by_eps = {}
binarization_eot_by_eps = {}
binarization_benchmarks_by_eps = {}

for eps_times in binarization_eps_values:
    plans = get_or_compute_binarization_plans(
        data_bundle,
        eps_times=eps_times,
        tol=tol,
        max_iter=max_iter,
    )
    binarization_plans_by_eps[float(eps_times)] = plans

    block_matrices = get_or_compute_binarization_block_matrices(plans, eps_times)
    binarization_block_matrices_by_eps[float(eps_times)] = block_matrices

    eot_reference = get_or_compute_binarization_eot_reference(
        data_bundle,
        plans,
        eps_times=eps_times,
        cache_dir=cache_dir,
        t=eot_t,
        embd_dim=eot_embd_dim,
        k_conc=k_conc,
        k_cross=k_cross,
    )
    binarization_eot_by_eps[float(eps_times)] = eot_reference

    cache_path = make_binarization_cache_path(cache_dir, eps_times)
    if cache_path.exists():
        benchmark = load_benchmark_cache(cache_path)
        benchmark['cache_path'] = str(cache_path)
        benchmark['graph_binarized'] = True
        benchmark['graph_symmetrized'] = True
        benchmark['graph_normalized'] = True
        print(f'Loaded binarization benchmark cache from {cache_path}')
    else:
        benchmark = run_top_pct_neighbors_benchmark(
            data_bundle,
            block_matrices,
            pct_values=pct_values,
            tsne_seeds=tsne_seeds,
            viz_seed=viz_seed,
            n_epochs=n_epochs,
            k_conc=k_conc,
            k_cross=k_cross,
            binarize_graph=True,
            symmetrize_graph=True,
            normalize_graph=True,
            cache_path=cache_path,
        )
        print(f'Saved binarization benchmark cache to {cache_path}')

    binarization_benchmarks_by_eps[float(eps_times)] = benchmark
    print(
        f"eps_times={eps_times:g}: cache={cache_path.name}, "
        f"ring EOT conc={eot_reference['ring']['metrics']['concordance']:.4f}, "
        f"gmm EOT conc={eot_reference['gmm']['metrics']['concordance']:.4f}"
    )

print(
    f'Prepared binarization benchmarks for eps_times={sorted(binarization_benchmarks_by_eps)} '
    f'with EOT t={eot_t} as reference.'
)


In [ ]:
binarization_preview_eps_times = 10.0
binarization_preview_pct = 10.0

preview_blocks = binarization_block_matrices_by_eps[float(binarization_preview_eps_times)]
binarization_preview_graphs = {}

for dataset_name in ('ring', 'gmm'):
    sparse_matrix = sparsify_top_percentage_neighbors_graph(
        preview_blocks[dataset_name]['matrix'],
        percentage=float(binarization_preview_pct) / 100.0,
        n0=data_bundle[dataset_name]['n0'],
        n1=data_bundle[dataset_name]['n1'],
    )
    graph = postprocess_graph_matrix(
        sparse_matrix,
        binarize=True,
        symmetrize=True,
        normalize=True,
    )
    binarization_preview_graphs[dataset_name] = {
        'sparse_matrix': sparse_matrix,
        'graph': graph,
    }
    print(
        f"{dataset_name}: sparse-matrix nnz={np.count_nonzero(sparse_matrix)}, "
        f"sparse-matrix sum={sparse_matrix.sum():.6f}, final graph nnz={np.count_nonzero(graph)}, "
        f"final graph sum={graph.sum():.6f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, dataset_name, panel_title in (
    (axes[0], 'ring', 'Ring'),
    (axes[1], 'gmm', 'GMM'),
):
    graph = binarization_preview_graphs[dataset_name]['graph']
    image = ax.imshow(
        graph,
        cmap='magma',
        interpolation='nearest',
        aspect='auto',
        vmin=0.0,
        vmax=float(graph.max()) if graph.size else 1.0,
    )
    ax.set_title(
        f"{panel_title} 4-block Step-5 binarized graph\n"
        f"(eps_times={binarization_preview_eps_times:g}, top {binarization_preview_pct:g}% neighbors)"
    )
    ax.set_xlabel('all points index')
    ax.set_ylabel('all points index')
    ax.grid(False)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
for eps_times in binarization_eps_values:
    benchmark = binarization_benchmarks_by_eps[float(eps_times)]
    fig, axes = plot_pct_embedding_grid(
        benchmark,
        data_bundle,
        title=(
            f"Step-5 binarized 4-block embeddings "
            f"(eps_times={benchmark['eps_times']:g}, viz seed={benchmark['viz_seed']})"
        ),
    )
    plt.show()


In [ ]:
for eps_times in binarization_eps_values:
    benchmark = binarization_benchmarks_by_eps[float(eps_times)]
    fig, axes = plot_gmm_pct_embedding_grid_by_cluster(
        benchmark,
        data_bundle,
        title=(
            f"Step-5 binarized 4-block GMM embeddings "
            f"(eps_times={benchmark['eps_times']:g}, viz seed={benchmark['viz_seed']}, colored by cluster)"
        ),
    )
    plt.show()


In [ ]:
for eps_times in binarization_eps_values:
    benchmark = binarization_benchmarks_by_eps[float(eps_times)]
    eot_reference = binarization_eot_by_eps[float(eps_times)]
    fig, axes = plot_k50_concordance_summary(
        benchmark,
        eot_reference,
        title=(
            f"Step-5 binarized 4-block summary "
            f"(eps_times={benchmark['eps_times']:g}, EOT t={eot_reference['t']})"
        ),
        x_label='Top % neighbors',
    )
    plt.show()


In [ ]:
binarization_metric_names = ('concordance', 'cross_concordance')
binarization_heatmap_data_by_metric = {}

for metric_name in binarization_metric_names:
    binarization_heatmap_data_by_metric[metric_name] = build_metric_heatmap_data(
        benchmarks_by_eps=binarization_benchmarks_by_eps,
        eot_references_by_eps=binarization_eot_by_eps,
        metric_name=metric_name,
    )

binarization_scale_bounds = compute_shared_dataset_scale_bounds(
    binarization_heatmap_data_by_metric
)
binarization_figure_dir = cache_dir / 'figures'
binarization_figure_dir.mkdir(parents=True, exist_ok=True)

for metric_name, heatmap_data in binarization_heatmap_data_by_metric.items():
    figure_path = binarization_figure_dir / (
        f'{metric_name}_eps_pct_heatmaps_4block_step5_binarized.png'
    )
    fig, axes = plot_metric_heatmap_grid(
        heatmap_data,
        title=(
            f"{heatmap_data['metric_label']} heatmaps "
            f"(4-block Step 5 with binarization)"
        ),
        save_path=figure_path,
        scale_bounds_by_dataset=binarization_scale_bounds,
        annotate_cells=True,
    )
    plt.show()
    print(f'Saved {metric_name} binarization heatmap figure to {figure_path}')

binarization_heatmap_cache_path = cache_dir / 'eps_pct_metric_heatmaps_4block_step5_binarized.npz'
np.savez_compressed(
    binarization_heatmap_cache_path,
    eps_times=binarization_heatmap_data_by_metric['concordance']['eps_times'],
    pct_values=binarization_heatmap_data_by_metric['concordance']['pct_values'],
    dataset_names=np.asarray(binarization_heatmap_data_by_metric['concordance']['dataset_names']),
    metric_names=np.asarray(binarization_metric_names),
    eot_t=np.asarray(binarization_heatmap_data_by_metric['concordance']['eot_t'], dtype=int),
    x_labels=np.asarray(binarization_heatmap_data_by_metric['concordance']['x_labels']),
    y_labels=np.asarray(binarization_heatmap_data_by_metric['concordance']['y_labels']),
    concordance_ring=binarization_heatmap_data_by_metric['concordance']['datasets']['ring']['full_matrix'],
    concordance_gmm=binarization_heatmap_data_by_metric['concordance']['datasets']['gmm']['full_matrix'],
    cross_concordance_ring=binarization_heatmap_data_by_metric['cross_concordance']['datasets']['ring']['full_matrix'],
    cross_concordance_gmm=binarization_heatmap_data_by_metric['cross_concordance']['datasets']['gmm']['full_matrix'],
)
print(f'Saved binarization heatmap cache to {binarization_heatmap_cache_path}')


### Weighted vs Binarized Diagnostics

This section isolates the effect of Step-5 binarization on the **same sparsified 4-block graph**. For each top-% setting, it builds one sparsified 4-block matrix and compares two postprocessing choices on identical support:

- `weighted`: symmetrize + normalize, while keeping the retained edge weights
- `binarized`: binarize + symmetrize + normalize

The diagnostics below make it easier to see whether higher `%` is mainly adding many weak edges that become equally strong after binarization.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ne_aligned_utils.benchmarking import (
    build_four_block_matrices,
    compute_ot_plans_for_eps,
    postprocess_graph_matrix,
    prepare_benchmark_data,
    sparsify_top_percentage_neighbors_graph,
)

diagnostics_eps_times = float(globals().get('diagnostics_eps_times', 300.0))
diagnostics_hist_pct_requested = float(globals().get('diagnostics_hist_pct', 30.0))
diagnostics_pct_values = [float(p) for p in globals().get('pct_values', [0.1, 0.5, 1, 5, 10, 12, 15, 17, 20, 22, 25, 30, 35, 40])]

diagnostics_seed = int(globals().get('seed', 0))
diagnostics_tol = float(globals().get('tol', 1e-12))
diagnostics_max_iter = int(globals().get('max_iter', int(1e4)))

n0_ring = int(globals().get('n0_ring', 1000))
n1_ring = int(globals().get('n1_ring', 5000))
gmm_n0 = int(globals().get('gmm_n0', 600))
gmm_n1 = int(globals().get('gmm_n1', 600))
gmm_p = int(globals().get('gmm_p', 1000))
gmm_q = int(globals().get('gmm_q', 6))
gmm_theta = float(globals().get('gmm_theta', 3.0))

if 'data_bundle' not in globals():
    data_bundle = prepare_benchmark_data(
        seed=diagnostics_seed,
        n0_ring=n0_ring,
        n1_ring=n1_ring,
        gmm_n0=gmm_n0,
        gmm_n1=gmm_n1,
        gmm_p=gmm_p,
        gmm_q=gmm_q,
        gmm_theta=gmm_theta,
    )


def get_or_compute_diagnostics_block_matrices(data_bundle, eps_times, tol, max_iter):
    if 'binarization_block_matrices_by_eps' in globals() and float(eps_times) in binarization_block_matrices_by_eps:
        return binarization_block_matrices_by_eps[float(eps_times)]
    if 'no_binarization_block_matrices_by_eps' in globals() and float(eps_times) in no_binarization_block_matrices_by_eps:
        return no_binarization_block_matrices_by_eps[float(eps_times)]

    block_name_by_eps = {
        1.0: 'block_matrices_1',
        10.0: 'block_matrices_10',
        100.0: 'block_matrices_100',
        300.0: 'block_matrices_1000',
    }
    plan_name_by_eps = {
        1.0: 'plans_1',
        10.0: 'plans_10',
        100.0: 'plans_100',
        300.0: 'plans_1000',
    }

    block_name = block_name_by_eps.get(float(eps_times))
    if block_name is not None and block_name in globals():
        return globals()[block_name]

    plan_name = plan_name_by_eps.get(float(eps_times))
    if plan_name is not None and plan_name in globals():
        plans = globals()[plan_name]
    else:
        plans = compute_ot_plans_for_eps(
            data_bundle,
            eps_times=eps_times,
            tol=tol,
            max_iter=max_iter,
        )
        if plan_name is not None:
            globals()[plan_name] = plans

    block_matrices = build_four_block_matrices(plans)
    if block_name is not None:
        globals()[block_name] = block_matrices
    return block_matrices


def block_mass_fractions(graph, n0, n1):
    total = float(graph.sum())
    a00 = float(graph[:n0, :n0].sum())
    a01 = float(graph[:n0, n0:n0 + n1].sum())
    a10 = float(graph[n0:n0 + n1, :n0].sum())
    a11 = float(graph[n0:n0 + n1, n0:n0 + n1].sum())
    if total <= 0:
        return {
            'a00': np.nan,
            'a01': np.nan,
            'a10': np.nan,
            'a11': np.nan,
            'within': np.nan,
            'cross': np.nan,
        }
    return {
        'a00': a00 / total,
        'a01': a01 / total,
        'a10': a10 / total,
        'a11': a11 / total,
        'within': (a00 + a11) / total,
        'cross': (a01 + a10) / total,
    }


def effective_degree_distribution(graph):
    row_sums = graph.sum(axis=1)
    nonzero_rows = row_sums > 0
    if not np.any(nonzero_rows):
        return np.asarray([], dtype=float)
    row_probs = graph[nonzero_rows] / row_sums[nonzero_rows, None]
    return 1.0 / np.sum(row_probs ** 2, axis=1)


def summarize_distribution(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return {
            'q10': np.nan,
            'q50': np.nan,
            'q90': np.nan,
            'mean': np.nan,
        }
    q10, q50, q90 = np.quantile(values, [0.1, 0.5, 0.9])
    return {
        'q10': float(q10),
        'q50': float(q50),
        'q90': float(q90),
        'mean': float(values.mean()),
    }


def nearest_pct_value(requested_pct, pct_values):
    pct_values = [float(p) for p in pct_values]
    return min(pct_values, key=lambda pct: abs(pct - float(requested_pct)))


diagnostics_hist_pct = nearest_pct_value(diagnostics_hist_pct_requested, diagnostics_pct_values)
print(
    f'Using diagnostics eps_times={diagnostics_eps_times:g} and histogram pct={diagnostics_hist_pct:g}% '
    f'(requested {diagnostics_hist_pct_requested:g}%).'
)

In [ ]:
diagnostics_blocks = get_or_compute_diagnostics_block_matrices(
    data_bundle,
    eps_times=diagnostics_eps_times,
    tol=diagnostics_tol,
    max_iter=diagnostics_max_iter,
)

diagnostics_by_dataset = {}
diagnostics_histograms = {}

for dataset_name in ('ring', 'gmm'):
    n0 = int(data_bundle[dataset_name]['n0'])
    n1 = int(data_bundle[dataset_name]['n1'])
    matrix = diagnostics_blocks[dataset_name]['matrix']

    dataset_diag = {
        'pct_values': [],
        'weighted_q10': [],
        'weighted_q50': [],
        'weighted_q90': [],
        'binarized_level': [],
        'weighted_cross': [],
        'binarized_cross': [],
        'weighted_eff_q10': [],
        'weighted_eff_q50': [],
        'weighted_eff_q90': [],
        'binarized_eff_q10': [],
        'binarized_eff_q50': [],
        'binarized_eff_q90': [],
        'weighted_nnz': [],
        'binarized_nnz': [],
    }

    for pct in diagnostics_pct_values:
        sparse_matrix = sparsify_top_percentage_neighbors_graph(
            matrix,
            percentage=float(pct) / 100.0,
            n0=n0,
            n1=n1,
        )
        weighted_graph = postprocess_graph_matrix(
            sparse_matrix,
            binarize=False,
            symmetrize=True,
            normalize=True,
        )
        binarized_graph = postprocess_graph_matrix(
            sparse_matrix,
            binarize=True,
            symmetrize=True,
            normalize=True,
        )

        weighted_positive = weighted_graph[weighted_graph > 0]
        binarized_positive = binarized_graph[binarized_graph > 0]

        weighted_weight_stats = summarize_distribution(weighted_positive)
        weighted_eff_stats = summarize_distribution(effective_degree_distribution(weighted_graph))
        binarized_eff_stats = summarize_distribution(effective_degree_distribution(binarized_graph))
        weighted_masses = block_mass_fractions(weighted_graph, n0=n0, n1=n1)
        binarized_masses = block_mass_fractions(binarized_graph, n0=n0, n1=n1)

        dataset_diag['pct_values'].append(float(pct))
        dataset_diag['weighted_q10'].append(weighted_weight_stats['q10'])
        dataset_diag['weighted_q50'].append(weighted_weight_stats['q50'])
        dataset_diag['weighted_q90'].append(weighted_weight_stats['q90'])
        dataset_diag['binarized_level'].append(
            float(binarized_positive[0]) if binarized_positive.size else np.nan
        )
        dataset_diag['weighted_cross'].append(weighted_masses['cross'])
        dataset_diag['binarized_cross'].append(binarized_masses['cross'])
        dataset_diag['weighted_eff_q10'].append(weighted_eff_stats['q10'])
        dataset_diag['weighted_eff_q50'].append(weighted_eff_stats['q50'])
        dataset_diag['weighted_eff_q90'].append(weighted_eff_stats['q90'])
        dataset_diag['binarized_eff_q10'].append(binarized_eff_stats['q10'])
        dataset_diag['binarized_eff_q50'].append(binarized_eff_stats['q50'])
        dataset_diag['binarized_eff_q90'].append(binarized_eff_stats['q90'])
        dataset_diag['weighted_nnz'].append(int(weighted_positive.size))
        dataset_diag['binarized_nnz'].append(int(binarized_positive.size))

        if np.isclose(float(pct), diagnostics_hist_pct):
            diagnostics_histograms[dataset_name] = {
                'weighted_positive': weighted_positive.copy(),
                'binarized_level': float(binarized_positive[0]) if binarized_positive.size else np.nan,
                'weighted_nnz': int(weighted_positive.size),
                'binarized_nnz': int(binarized_positive.size),
            }

    diagnostics_by_dataset[dataset_name] = dataset_diag

for dataset_name in ('ring', 'gmm'):
    dataset_diag = diagnostics_by_dataset[dataset_name]
    final_idx = len(dataset_diag['pct_values']) - 1
    hist_idx = dataset_diag['pct_values'].index(float(diagnostics_hist_pct))
    print(
        f"{dataset_name}: pct={diagnostics_hist_pct:g}% weighted q10={dataset_diag['weighted_q10'][hist_idx]:.3e}, "
        f"q50={dataset_diag['weighted_q50'][hist_idx]:.3e}, q90={dataset_diag['weighted_q90'][hist_idx]:.3e}, "
        f"binarized level={dataset_diag['binarized_level'][hist_idx]:.3e}"
    )
    print(
        f"{dataset_name}: pct={dataset_diag['pct_values'][final_idx]:g}% cross-mass weighted={dataset_diag['weighted_cross'][final_idx]:.3f}, "
        f"binarized={dataset_diag['binarized_cross'][final_idx]:.3f}; "
        f"effective-degree median weighted={dataset_diag['weighted_eff_q50'][final_idx]:.2f}, "
        f"binarized={dataset_diag['binarized_eff_q50'][final_idx]:.2f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

for col, dataset_name in enumerate(('ring', 'gmm')):
    hist_info = diagnostics_histograms[dataset_name]
    weighted_positive = hist_info['weighted_positive']
    binarized_level = hist_info['binarized_level']
    ax = axes[0, col]
    if weighted_positive.size:
        ax.hist(np.log10(weighted_positive), bins=30, color='steelblue', alpha=0.8)
    if np.isfinite(binarized_level) and binarized_level > 0:
        ax.axvline(np.log10(binarized_level), color='crimson', linestyle='--', linewidth=2)
    ax.set_title(f"{dataset_name} edge-weight spread at {diagnostics_hist_pct:g}%")
    ax.set_xlabel('log10(final positive edge weight)')
    ax.set_ylabel('count')
    ax.text(
        0.02,
        0.98,
        (
            f"weighted nnz={hist_info['weighted_nnz']}\n"
            f"binarized nnz={hist_info['binarized_nnz']}"
        ),
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=9,
    )

    dataset_diag = diagnostics_by_dataset[dataset_name]
    pcts = dataset_diag['pct_values']
    ax = axes[1, col]
    ax.plot(pcts, dataset_diag['weighted_q10'], color='steelblue', linestyle=':', label='weighted q10')
    ax.plot(pcts, dataset_diag['weighted_q50'], color='steelblue', linestyle='-', label='weighted q50')
    ax.plot(pcts, dataset_diag['weighted_q90'], color='steelblue', linestyle='--', label='weighted q90')
    ax.plot(pcts, dataset_diag['binarized_level'], color='crimson', linewidth=2, label='binarized level')
    ax.set_yscale('log')
    ax.set_title(f"{dataset_name} positive weights across top-%")
    ax.set_xlabel('top % neighbors')
    ax.set_ylabel('final edge weight')
    ax.grid(alpha=0.25)
    ax.legend(loc='best', fontsize=8)

fig.suptitle(
    f"Weighted vs binarized edge strengths on the same sparsified 4-block graph (eps_times={diagnostics_eps_times:g})",
    fontsize=14,
)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

for col, dataset_name in enumerate(('ring', 'gmm')):
    dataset_diag = diagnostics_by_dataset[dataset_name]
    pcts = np.asarray(dataset_diag['pct_values'], dtype=float)

    ax = axes[0, col]
    ax.plot(pcts, dataset_diag['weighted_cross'], color='steelblue', linewidth=2, label='weighted cross-block mass')
    ax.plot(pcts, dataset_diag['binarized_cross'], color='crimson', linewidth=2, label='binarized cross-block mass')
    ax.set_ylim(0.0, 1.0)
    ax.set_title(f"{dataset_name} cross-block mass fraction")
    ax.set_xlabel('top % neighbors')
    ax.set_ylabel('cross-block mass / total mass')
    ax.grid(alpha=0.25)
    ax.legend(loc='best', fontsize=8)

    ax = axes[1, col]
    ax.fill_between(
        pcts,
        dataset_diag['weighted_eff_q10'],
        dataset_diag['weighted_eff_q90'],
        color='steelblue',
        alpha=0.15,
    )
    ax.fill_between(
        pcts,
        dataset_diag['binarized_eff_q10'],
        dataset_diag['binarized_eff_q90'],
        color='crimson',
        alpha=0.15,
    )
    ax.plot(pcts, dataset_diag['weighted_eff_q50'], color='steelblue', linewidth=2, label='weighted median')
    ax.plot(pcts, dataset_diag['binarized_eff_q50'], color='crimson', linewidth=2, label='binarized median')
    ax.set_title(f"{dataset_name} row effective degree")
    ax.set_xlabel('top % neighbors')
    ax.set_ylabel(r'$1 / \sum_j p_{ij}^2$ after row normalization')
    ax.grid(alpha=0.25)
    ax.legend(loc='best', fontsize=8)

fig.suptitle(
    f"Why larger top-% can hurt after binarization (eps_times={diagnostics_eps_times:g})",
    fontsize=14,
)
plt.show()